# Siamese UNet for S1GFloods Flood-Change Detection
### S1GFloods PNG training → real Sentinel-1 GeoTIFF deployment (Gaibandha)

## Change log / acceptance summary
This notebook is an update of the original Modified-Sen1Floods11 notebook. The overall workflow, model zoo, losses, metrics, DDP runner, bootstrap confidence intervals, qualitative/EPS exports, manifest, and zip export are retained, while the data and deployment paths are changed as follows:

1. Training now discovers the folder-defined **S1GFloods** `train/val/test × A/B/GT` PNG splits and checks A/B/GT filename parity; expected counts are 4300/530/530 (warnings only if counts differ).
2. Removed the Sen1Floods11 split-CSV download, official split reconstruction, Bolivia holdout, VV/VH GeoTIFF training reader, `-1` label masking, and percentile normalization.
3. Added a guarded `WIPE_WORKING=False` switch so `/kaggle/working` is not erased unless explicitly requested.
4. Added a PNG audit for channel identity, 0/255 clipping fractions, flood fraction, class-conditional gray medians, per-patch temporal land shift, density crossover, patch medians, and a heuristic both-zero border/straight-edge NoData audit. Audit tables are saved under `split_audit/`.
5. Added `S1GFloodsPreprocessor` as the single saved preprocessing definition. PNGs use `gray/255`; real VV dB GeoTIFFs use the checkpoint's fixed dB→gray mapping and optional 8-bit quantization.
6. Added `INPUT_MODE="1ch"` (default) and `"3ch_replicated"` (for the ImageNet-pretrained VMamba ablation); every model's input channels now derive from `IN_CHANNELS`.
7. Rewrote the dataset for PNG A/B/GT. Labels are `GT>127`; all pixels are valid unless `MASK_BOTH_ZERO=True`; PRE/POST/label/mask use exactly the same spatial transform.
8. Removed the duplicate radiometric-augmentation class. Training-only gray-domain augmentation now supports dark-tail compression, joint gain/offset, independent small per-date offsets, and joint Gaussian blur, each controlled by config flags. PRE/POST are never swapped because the target is asymmetric.
9. Added gradient accumulation (`ACCUM_STEPS`) while keeping AMP, validation-IoU model selection, and the existing loss zoo. `bce_dice` remains the default; rare-positive alternatives are only recommended, never silently selected.
10. Fixed validation-threshold persistence: the validation-selected threshold is written back into `best_model.pt` and also saved as JSON; GeoTIFF inference reads the locked checkpoint threshold.
11. Added optional flip/rotation TTA for both benchmark evaluation and GeoTIFF inference (`INFERENCE_TTA=False` by default).
12. Replaced the former Bolivia evaluation cell with optional external-reference evaluation (`EXTERNAL_REF_MASK`), with strict raster-grid compatibility and no resizing.
13. Rewrote GeoTIFF inference to read **VV by band name**, reject geometry mismatches, warn on non-10 m or geographic CRS, cast to float32, fill invalid footprint pixels safely, tile at 256 with 128 overlap, use Hann blending, preserve invalid pixels as NoData, and preserve CRS/transform.
14. Added scene-level median-gray NoData filling, safe reflect/constant edge padding, checkpoint preprocessing restoration, optional identical two-point external calibration (OFF by default), and inference metadata recording all unverified/diagnostic settings.
15. Fixed area computation: projected rasters use affine pixel area (with CRS unit factor when available); geographic rasters use latitude-dependent spherical row areas. Flood area is reported in km² and as % of valid area.
16. Added a mandatory 3-pair **GeoTIFF↔PNG parity test** that inverts S1GFloods gray→dB, writes synthetic VV/VH GeoTIFFs, runs the deployment function, and asserts max probability difference `<1e-2` without TTA.
17. Added a user-selected Gaibandha ROI radiometric-sensitivity test over four fixed `(lo,hi)` mappings and median/zero NoData filling, reporting area and pairwise mask IoU without tuning the mapping to improve results.
18. Rewrote the domain-shift report for S1GFloods gray distributions vs Gaibandha PRE/POST, including percentiles, 0/255 fractions, `<60`/`<99`, Wasserstein distance, linear/log histograms, and PRE/POST/difference figures.
19. Updated qualitative figures, model-complexity dummy inputs, results tables, experiment manifest, output names, and notebook instructions for single-VV S1GFloods.
20. Generation-time validation passed: **all 45 code cells pass `ast.parse`**, and the notebook passes `nbformat.validate`. Runtime GPU/Mamba checks remain environment-dependent.

## Verified dataset facts used by this notebook
- S1GFloods: 5,360 pairs, 256×256 at 10 m; split folders `train/val/test` with expected 4300/530/530 pairs.
- `A`=pre, `B`=post, `GT`=binary newly inundated flood; PNG A/B have 3 identical channels representing one VV band; there is no VH and no `-1` invalid class.
- Creator pipeline: sigma0 calibration → dB → fixed linear 8-bit scaling; no speckle filtering.

## UNVERIFIED assumptions — do not present these as dataset facts
- `PNG_DB_LO=-30.0`, `PNG_DB_HI=0.0` is an inferred, not author-confirmed, dB→gray mapping.
- `EXTERNAL_INPUT_IS_COMPOSITE=True` is an editable assumption about the Gaibandha exports.
- The observed Gaibandha radiometric shift means external flood recall may be underestimated even when benchmark metrics are strong.

### Main switches
- Change `INPUT_MODE` between `"1ch"` and `"3ch_replicated"`; `IN_CHANNELS` is derived automatically.
- Change `PNG_DB_LO/PNG_DB_HI` only as an explicitly reported assumption/sensitivity condition; never tune them on the external scene to make results look better.
- Keep `MASK_BOTH_ZERO=False` unless the both-zero border audit demonstrates straight-edged footprint NoData.
- Edit only values marked `# <<< EDIT THIS` unless you intentionally change the experiment design.


In [1]:
import os
import shutil

WIPE_WORKING = False  # <<< EDIT THIS — destructive; keep False unless you explicitly want a clean /kaggle/working
WORKING_DIR = "/kaggle/working"

if WIPE_WORKING:
    print(f"WIPE_WORKING=True: deleting contents of {WORKING_DIR}")
    for filename in os.listdir(WORKING_DIR):
        file_path = os.path.join(WORKING_DIR, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            print(f"Failed to delete {file_path}. Reason: {e}")
else:
    print("WIPE_WORKING=False: preserving existing /kaggle/working contents.")

WIPE_WORKING=False: preserving existing /kaggle/working contents.


In [2]:
# Cell 1A — S1GFloods uses folder-defined train/val/test splits; no Sen1Floods11 CSV download is needed.
print("S1GFloods split source: DATA_ROOT/train, DATA_ROOT/val, DATA_ROOT/test (A/B/GT PNG folders).")
print("Expected pair counts: train=4300, val=530, test=530. Count mismatches warn but do not crash.")

S1GFloods split source: DATA_ROOT/train, DATA_ROOT/val, DATA_ROOT/test (A/B/GT PNG folders).
Expected pair counts: train=4300, val=530, test=530. Count mismatches warn but do not crash.


In [3]:
# Cell 2 — Install dependencies
# Kaggle images already ship torch/torchvision with CUDA, so we do NOT reinstall torch (that risks breaking the GPU build).
# We only add the extra geospatial / metrics / plotting packages we need.
import sys, subprocess

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps"] + pkgs, check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)

REQUIRED = [
    "rasterio>=1.3.9",
    "scikit-learn>=1.3.0",
    "scikit-image>=0.22.0",
    "albumentations>=1.4.0",
    "tifffile>=2023.9.26",
    "pandas>=2.0.0",
    "matplotlib>=3.7.0",
    "seaborn>=0.13.0",
    "tqdm>=4.66.0",
    "openpyxl>=3.1.0",
    "thop>=0.1.1.post2209072238",   # FLOPs/MACs estimation
    "einops>=0.7.0",                # tensor reshaping for BIT / ChangeFormer attention
]

try:
    pip_install(REQUIRED)
    print("Dependency installation finished.")
except subprocess.CalledProcessError as e:
    print("WARNING: some packages failed to install via pip. Notebook will still run; "
          "optional features (e.g. FLOPs counting) may be skipped.", e)

Dependency installation finished.


In [4]:
# # --- ChangeMamba (MambaCD) setup ---------------------------------------------------
# import subprocess as _sp
# import os

# CHANGEMAMBA_DIR = "/kaggle/working/ChangeMamba"   # <<< EDIT THIS if you want it elsewhere

# if not os.path.isdir(CHANGEMAMBA_DIR):
#     _sp.run(["git", "clone", "--depth", "1",
#               "https://github.com/ChenHongruixuan/ChangeMamba.git", CHANGEMAMBA_DIR], check=True)

# # Install its Python deps, but SKIP torch/torchvision/torchaudio/triton — Kaggle's
# # preinstalled CUDA-matched build must not be overwritten (same rule as the REQUIRED list above).
# req_path = os.path.join(CHANGEMAMBA_DIR, "requirements.txt")
# skip = {"torch", "torchvision", "torchaudio", "triton"}
# with open(req_path) as f:
#     cm_pkgs = [l.strip() for l in f if l.strip() and l.strip().split("==")[0].split(">=")[0] not in skip]
# _sp.run([sys.executable, "-m", "pip", "install", "-q"] + cm_pkgs, check=True)

# # Build the custom selective-scan CUDA kernel (needs nvcc matching torch's CUDA build; this is
# # the main thing that can fail on a given Kaggle image — if it does, check `nvcc --version`
# # vs `torch.version.cuda`).
# kernel_dir = os.path.join(CHANGEMAMBA_DIR, "kernels", "selective_scan")
# _sp.run([sys.executable, "-m", "pip", "install", "-q", "."], cwd=kernel_dir, check=True)

# sys.path.insert(0, CHANGEMAMBA_DIR)
# print("ChangeMamba repo ready at", CHANGEMAMBA_DIR)

In [5]:
# import subprocess, sys, os

# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
# import gdown

# os.makedirs("/kaggle/working/ChangeMamba/pretrained_weight", exist_ok=True)
# gdown.download(
#     id="160PXughGMNZ1GyByspLFS68sfUdrQE2N",
#     output="/kaggle/working/ChangeMamba/pretrained_weight/vssm_tiny_imagenet.pth",
#     quiet=False,
# )

In [6]:
# Cell 3 — Imports
import os, re, io, json, glob, math, time, random, shutil, zipfile, warnings, platform, subprocess, getpass
import hashlib
import contextlib
from pathlib import Path
from dataclasses import dataclass, field, asdict
from collections import defaultdict, OrderedDict
from typing import Optional, List, Tuple, Dict, Any

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.ndimage import gaussian_filter, gaussian_filter1d, label as cc_label
from scipy.stats import wasserstein_distance
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import torch.multiprocessing as mp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as TorchDDP
from torch.utils.data.distributed import DistributedSampler

import einops
from einops import rearrange

import rasterio
from rasterio.windows import Window
from rasterio.transform import Affine, from_origin

from sklearn.metrics import (
    roc_auc_score, average_precision_score, cohen_kappa_score,
    matthews_corrcoef, balanced_accuracy_score, confusion_matrix
)

warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)
sns.set_theme(style="whitegrid")
print("Imports OK.")

Imports OK.


In [7]:
# Cell 4 — Central configuration. Edit paths/settings marked "<<< EDIT THIS".
# Every downstream cell reads from CONFIG / these constants; do not hardcode experiment settings elsewhere.

# ---- Paths --------------------------------------------------------------
DATA_ROOT = "/kaggle/input/datasets/ashrifrahman/s1gfloods"  # <<< EDIT THIS
OUTPUT_ROOT = "/kaggle/working/changeMamba_s1gfloods"

EXTERNAL_PRE_TIF = "/kaggle/input/datasets/ashrifrahman/gaibandha-sar/SPARRSO/Gaibandha_S1_pre_June 8_20 2024.tif"  # <<< EDIT THIS
EXTERNAL_POST_TIF = "/kaggle/input/datasets/ashrifrahman/gaibandha-sar/SPARRSO/Gaibandha_S1_post_June 25_July 10_2024.tif"  # <<< EDIT THIS
EXTERNAL_REF_MASK = None  # <<< EDIT THIS — optional reference-mask GeoTIFF; keep None if unavailable
EXTERNAL_REF_POSITIVE_VALUE = None  # <<< EDIT THIS — None means any valid value > 0 is positive

# ---- Dataset / input domain --------------------------------------------
INPUT_MODE = "1ch"  # <<< EDIT THIS — "1ch" | "3ch_replicated"
if INPUT_MODE not in {"1ch", "3ch_replicated"}:
    raise ValueError("INPUT_MODE must be '1ch' or '3ch_replicated'.")
IN_CHANNELS = 1 if INPUT_MODE == "1ch" else 3

# UNVERIFIED: inferred from measured S1GFloods gray statistics; not confirmed by the dataset authors.
PNG_DB_LO = -30.0  # <<< EDIT THIS
PNG_DB_HI = 0.0    # <<< EDIT THIS
PNG_QUANTIZE = True
MASK_BOTH_ZERO = False  # <<< EDIT THIS — enable only if the zero-border audit supports footprint NoData masking

# ---- Experiment identity ----------------------------------------------
MODEL_NAME = "siamese_unet"  # bitemporal_unet | siamese_unet | fc_siam_conc | fc_siam_diff | attention_unet | bit | changeformer | changemamba
TARGET_MODE = "newly_inundated_flood"  # S1GFloods GT semantics supplied by the user

# ---- ChangeMamba (MambaCD) --------------------------------------------
CHANGEMAMBA_VARIANT = "tiny"  # "tiny" | "small" | "base"
CHANGEMAMBA_ENCODER_PRETRAINED = None  # <<< EDIT THIS — use INPUT_MODE="3ch_replicated" for ImageNet VMamba weights
_CHANGEMAMBA_VARIANTS = {
    "tiny":  ([2, 2, 4, 2], 96),
    "small": ([2, 2, 15, 2], 96),
    "base":  ([2, 2, 15, 2], 128),
}
if CHANGEMAMBA_ENCODER_PRETRAINED is not None and IN_CHANNELS != 3:
    raise ValueError("ImageNet-pretrained VMamba weights require INPUT_MODE='3ch_replicated' (IN_CHANNELS=3).")

# ---- Training hyperparameters -----------------------------------------
# Reference defaults requested for S1GFloods: 256 patch, batch ~16 effective, lr=1e-4, 100 epochs, hybrid loss.
EPOCHS = 200
BATCH_SIZE_PER_GPU = 4  # <<< EDIT THIS — try 4-8 for ChangeMamba at 256×256 if memory allows
ACCUM_STEPS = 4         # <<< EDIT THIS — 4×4=16 on one GPU; use 2 with two-GPU DDP for global effective batch 16
NUM_WORKERS = 0

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
SCHEDULER_NAME = "cosine"
WARMUP_EPOCHS = 5

LOSS_NAME = "bce_dice"  # bce | dice | bce_dice | focal | focal_dice | tversky | bce_tversky | focal_tversky
BCE_WEIGHT = 0.5
DICE_WEIGHT = 0.5
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
TVERSKY_ALPHA = 0.3
TVERSKY_BETA = 0.7
EARLY_STOPPING_PATIENCE = 50
CHECKPOINT_METRIC = "val_iou"

SEED = 42
RUN_SEEDS = [42]  # <<< EDIT THIS to [42, 52, 62, 72, 82] for final multi-seed reporting
USE_AMP = True
USE_DDP = True
USE_AUGMENTATION = True
USE_SAR_RADIOMETRIC_AUG = True

# Gray-domain radiometric robustness controls (training only).
AUG_DARK_TAIL = True
AUG_GAIN_OFFSET = True
AUG_DATE_OFFSET = True
AUG_BLUR = True

# ---- BIT ---------------------------------------------------------------
BIT_DIM = 32
BIT_TOKENS = 4
BIT_ENC_DEPTH = 1
BIT_DEC_DEPTH = 8
BIT_HEADS = 8

# ---- ChangeFormer ------------------------------------------------------
CHANGEFORMER_EMBED_DIMS = [32, 64, 160, 256]
CHANGEFORMER_DEPTHS = [2, 2, 2, 2]
CHANGEFORMER_HEADS = [1, 2, 5, 8]
CHANGEFORMER_DECODER_DIM = 256

# ---- Inference ---------------------------------------------------------
INFERENCE_TILE_SIZE = 256
INFERENCE_OVERLAP = 128
INFERENCE_TTA = False
AUTO_ALIGN = False
NODATA_FILL = "median"  # <<< EDIT THIS — "median" | "zero"
EXTERNAL_INPUT_IS_COMPOSITE = True  # <<< EDIT THIS — UNVERIFIED assumption written to metadata

# Optional external two-point calibration; OFF by default and approximate when used.
EXTERNAL_CALIBRATION = "none"  # <<< EDIT THIS — "none" | "two_point"
CALIBRATION_STABLE_LAND_DB = None  # <<< EDIT THIS if EXTERNAL_CALIBRATION="two_point"
CALIBRATION_RIVER_DB = None        # <<< EDIT THIS if EXTERNAL_CALIBRATION="two_point"
CALIBRATION_TARGET_LAND_GRAY = 144.0
CALIBRATION_TARGET_WATER_GRAY = 47.0

# User-selected external ROI for the radiometric sensitivity test: (col_off, row_off, width, height).
ROI_WINDOW = None  # <<< EDIT THIS — e.g. (1000, 1500, 1024, 1024)
SENSITIVITY_DB_RANGES = [(-30.0, 0.0), (-28.0, -2.0), (-32.0, -2.0), (-30.0, -2.0)]
SENSITIVITY_NODATA_FILLS = ["median", "zero"]
PARITY_TEST_PAIRS = 3
PARITY_TOLERANCE = 1e-2
DOMAIN_FIGURE_MAX_SIZE = 1024

# ---- Threshold ---------------------------------------------------------
THRESHOLD = 0.5
THRESHOLD_SEARCH_RANGE = (0.10, 0.90)
THRESHOLD_SEARCH_STEPS = 33

# ---- Resume / reporting ------------------------------------------------
RESUME_CHECKPOINT = None  # <<< EDIT THIS
BOOTSTRAP_REPS = 1000
BOOTSTRAP_CI = 0.95

print("UNVERIFIED: PNG dB→gray mapping assumed linear over", (PNG_DB_LO, PNG_DB_HI))
print("UNVERIFIED: EXTERNAL_INPUT_IS_COMPOSITE =", EXTERNAL_INPUT_IS_COMPOSITE)
print("Input mode:", INPUT_MODE, "| IN_CHANNELS:", IN_CHANNELS)


def _is_json_serializable(v):
    try:
        json.dumps(v)
        return True
    except (TypeError, ValueError):
        return False

CONFIG: Dict[str, Any] = {
    k: v for k, v in dict(globals()).items()
    if k.isupper() and not k.startswith("_") and k not in ("CONFIG",)
    and _is_json_serializable(v)
}
print(f"Loaded {len(CONFIG)} config keys.")

UNVERIFIED: PNG dB→gray mapping assumed linear over (-30.0, 0.0)
UNVERIFIED: EXTERNAL_INPUT_IS_COMPOSITE = True
Input mode: 1ch | IN_CHANNELS: 1
Loaded 79 config keys.


In [8]:
# Cell 5 — Reproducibility
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    # Deterministic where practical. Full determinism on cuDNN convs costs throughput; we trade a little
    # determinism for speed on the (already algorithmically-controlled) conv backend but fix everything else.
    torch.backends.cudnn.benchmark = True
    torch.use_deterministic_algorithms(False)  # True is safer for bit-exactness but breaks some cuDNN kernels/ops we use
    return seed

set_global_seed(SEED)
print(f"Global seed set to {SEED}")


Global seed set to 42


In [9]:
# Cell 6 — GPU / environment info
def print_environment_info():
    print("="*70)
    print("Python       :", platform.python_version())
    print("PyTorch      :", torch.__version__)
    print("CUDA (torch) :", torch.version.cuda)
    print("cuDNN        :", torch.backends.cudnn.version())
    print("CUDA available:", torch.cuda.is_available())
    n_gpu = torch.cuda.device_count()
    print("GPU count    :", n_gpu)
    for i in range(n_gpu):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB, SM {p.major}.{p.minor}")
    print("="*70)
    return n_gpu

N_GPUS = print_environment_info()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#EFFECTIVE_DDP = USE_DDP and N_GPUS > 1
EFFECTIVE_DDP = False
#EFFECTIVE_DP = (not EFFECTIVE_DDP) and N_GPUS > 1
EFFECTIVE_DP = False
print(f"Strategy -> DDP: {EFFECTIVE_DDP} | DataParallel: {EFFECTIVE_DP} | single-device: {not (EFFECTIVE_DDP or EFFECTIVE_DP)}")


Python       : 3.12.13
PyTorch      : 2.10.0+cu128
CUDA (torch) : 12.8
cuDNN        : 91002
CUDA available: True
GPU count    : 2
  GPU 0: Tesla T4, 15.6 GB, SM 7.5
  GPU 1: Tesla T4, 15.6 GB, SM 7.5
Strategy -> DDP: False | DataParallel: False | single-device: True


In [10]:
# Cell 7 — Output directory structure
SUBDIRS = ["checkpoints", "logs", "metrics", "figures", "predictions", "qualitative",
           "configs", "inference_outputs", "environment", "split_audit"]

def make_output_dirs(root: str) -> Dict[str, str]:
    paths = {}
    os.makedirs(root, exist_ok=True)
    for d in SUBDIRS:
        p = os.path.join(root, d)
        os.makedirs(p, exist_ok=True)
        paths[d] = p
    return paths

DIRS = make_output_dirs(OUTPUT_ROOT)

with open(os.path.join(DIRS["configs"], "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

print("Output root:", OUTPUT_ROOT)
for k, v in DIRS.items():
    print(f"  {k:<20s} -> {v}")


Output root: /kaggle/working/changeMamba_s1gfloods
  checkpoints          -> /kaggle/working/changeMamba_s1gfloods/checkpoints
  logs                 -> /kaggle/working/changeMamba_s1gfloods/logs
  metrics              -> /kaggle/working/changeMamba_s1gfloods/metrics
  figures              -> /kaggle/working/changeMamba_s1gfloods/figures
  predictions          -> /kaggle/working/changeMamba_s1gfloods/predictions
  qualitative          -> /kaggle/working/changeMamba_s1gfloods/qualitative
  configs              -> /kaggle/working/changeMamba_s1gfloods/configs
  inference_outputs    -> /kaggle/working/changeMamba_s1gfloods/inference_outputs
  environment          -> /kaggle/working/changeMamba_s1gfloods/environment
  split_audit          -> /kaggle/working/changeMamba_s1gfloods/split_audit


In [11]:
# Cell 8 — S1GFloods flat-folder discovery + deterministic split
# Dataset layout:
# DATA_ROOT/A/
# DATA_ROOT/B/
# DATA_ROOT/Label/
#
# IMPORTANT:
# No official split metadata is available in this dataset copy.
# Therefore a deterministic natural-filename split is constructed here.
# Label/ is treated logically as GT/; files are NOT renamed or copied.

from pathlib import Path
import re
import pandas as pd

EXPECTED_COUNTS = {
    "train": 4300,
    "val": 530,
    "test": 530,
}

VALID_SPLITS = ("train", "val", "test")


def natural_key(name):
    """
    Natural filename ordering:
    image_2.png comes before image_10.png.
    """
    return [
        int(x) if x.isdigit() else x.lower()
        for x in re.split(r"(\d+)", str(name))
    ]


def find_role_folder(root, possible_names):
    """
    Case-insensitive folder lookup.
    For example Label, label, LABEL are all accepted.
    """
    root = Path(root)

    dirs = {
        p.name.lower(): p
        for p in root.iterdir()
        if p.is_dir()
    }

    for name in possible_names:
        if name.lower() in dirs:
            return dirs[name.lower()]

    raise FileNotFoundError(
        f"Could not find any of {possible_names} under {root}. "
        f"Found folders: {[p.name for p in root.iterdir() if p.is_dir()]}"
    )


def png_map(folder):
    """Map filename -> full path."""
    files = list(folder.glob("*.png")) + list(folder.glob("*.PNG"))

    mapping = {}
    for p in files:
        if p.name in mapping:
            raise RuntimeError(f"Duplicate filename detected: {p.name}")
        mapping[p.name] = str(p)

    return mapping


def discover_and_split_s1gfloods(root):
    root = Path(root)

    if not root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {root}")

    # ---------------------------------------------------------
    # Locate A / B / Label
    # Label is treated as GT logically; nothing is renamed.
    # ---------------------------------------------------------
    a_dir = find_role_folder(root, ["A"])
    b_dir = find_role_folder(root, ["B"])
    label_dir = find_role_folder(root, ["Label", "GT"])

    print("Detected folders:")
    print("  PRE   :", a_dir)
    print("  POST  :", b_dir)
    print("  LABEL :", label_dir)
    print("  NOTE  : Label/ is treated as GT; files are not renamed.")

    # ---------------------------------------------------------
    # Collect filenames
    # ---------------------------------------------------------
    a_files = png_map(a_dir)
    b_files = png_map(b_dir)
    label_files = png_map(label_dir)

    a_names = set(a_files)
    b_names = set(b_files)
    l_names = set(label_files)

    # Strict pairing check
    if not (a_names == b_names == l_names):
        print("\nFilename mismatch detected:")
        print("A only:", sorted(a_names - b_names - l_names)[:10])
        print("B only:", sorted(b_names - a_names - l_names)[:10])
        print("Label only:", sorted(l_names - a_names - b_names)[:10])

        raise RuntimeError(
            "A, B and Label filenames do not match exactly. "
            "Fix the dataset before training."
        )

    # Natural deterministic ordering
    names = sorted(a_names, key=natural_key)

    n_total = len(names)

    print(f"\nTotal matched A/B/Label pairs: {n_total}")

    # ---------------------------------------------------------
    # Preferred split: 4300 / 530 / 530
    # ---------------------------------------------------------
    expected_total = sum(EXPECTED_COUNTS.values())  # 5360

    if n_total == expected_total:
        n_train = EXPECTED_COUNTS["train"]
        n_val = EXPECTED_COUNTS["val"]

    else:
        # Fallback using approximately the same proportions
        # 4300/5360 ≈ 80.22%
        # 530/5360  ≈ 9.89%
        print(
            f"WARNING: expected {expected_total} images but found {n_total}.\n"
            "Using proportional deterministic split instead."
        )

        train_ratio = 4300 / 5360
        val_ratio = 530 / 5360

        n_train = int(round(n_total * train_ratio))
        n_val = int(round(n_total * val_ratio))

    train_names = names[:n_train]
    val_names = names[n_train:n_train + n_val]
    test_names = names[n_train + n_val:]

    split_names = {
        "train": train_names,
        "val": val_names,
        "test": test_names,
    }

    # ---------------------------------------------------------
    # Construct EXACT structure expected by rest of notebook
    # ---------------------------------------------------------
    splits = {}

    summary_rows = []

    for split in VALID_SPLITS:

        split_list = []

        for name in split_names[split]:

            split_list.append({
                "pre": a_files[name],
                "post": b_files[name],
                "label": label_files[name],
                "key": f"{split}/{name}",
            })

        splits[split] = split_list

        summary_rows.append({
            "split": split,
            "count": len(split_list),
        })

    summary_df = pd.DataFrame(summary_rows)

    # ---------------------------------------------------------
    # Safety checks
    # ---------------------------------------------------------
    train_set = set(train_names)
    val_set = set(val_names)
    test_set = set(test_names)

    assert train_set.isdisjoint(val_set)
    assert train_set.isdisjoint(test_set)
    assert val_set.isdisjoint(test_set)

    assert (
        len(train_set) +
        len(val_set) +
        len(test_set)
        == n_total
    )

    print("\nConstructed deterministic split:")
    print(summary_df.to_string(index=False))

    print("\nFirst/last filenames:")
    for split in VALID_SPLITS:
        ns = split_names[split]
        if ns:
            print(
                f"{split:5s}: "
                f"{ns[0]}  ->  {ns[-1]}"
            )

    print(
        "\nIMPORTANT CAVEAT:\n"
        "No official split metadata or event/region identifiers were available "
        "in this dataset copy. Train/validation/test sets were therefore "
        "constructed deterministically from natural filename order. "
        "Event-level or geographic independence cannot be verified."
    )

    return splits, summary_df


# =============================================================
# Build SPLITS used by all downstream notebook cells
# =============================================================

SPLITS, split_summary = discover_and_split_s1gfloods(DATA_ROOT)

split_source = (
    "Deterministic natural-filename split from flat "
    "A/B/Label folders; no official split metadata available"
)

triplet_by_key = {
    t["key"]: t
    for split in VALID_SPLITS
    for t in SPLITS[split]
}

# Final confirmation
print("\nFinal SPLITS:")
for split in VALID_SPLITS:
    print(f"{split}: {len(SPLITS[split])}")

Detected folders:
  PRE   : /kaggle/input/datasets/ashrifrahman/s1gfloods/A
  POST  : /kaggle/input/datasets/ashrifrahman/s1gfloods/B
  LABEL : /kaggle/input/datasets/ashrifrahman/s1gfloods/Label
  NOTE  : Label/ is treated as GT; files are not renamed.

Total matched A/B/Label pairs: 5360

Constructed deterministic split:
split  count
train   4300
  val    530
 test    530

First/last filenames:
train: image_1.png  ->  image_4300.png
val  : image_4301.png  ->  image_4830.png
test : image_4831.png  ->  image_5360.png

IMPORTANT CAVEAT:
No official split metadata or event/region identifiers were available in this dataset copy. Train/validation/test sets were therefore constructed deterministically from natural filename order. Event-level or geographic independence cannot be verified.

Final SPLITS:
train: 4300
val: 530
test: 530


In [12]:
# Cell 9 — Split verification / audit export
# S1GFloods ships train/val/test folders directly; there is no Sen1Floods11 split manifest or Bolivia reconstruction.

split_summary.to_csv(os.path.join(DIRS["split_audit"], "split_summary.csv"), index=False)
print(split_summary.to_string(index=False))
print("Split source:", split_source)
print(
    "Split method: deterministic natural-filename ordering. "
    "No random split is used."
)

split  count
train   4300
  val    530
 test    530
Split source: Deterministic natural-filename split from flat A/B/Label folders; no official split metadata available
Split method: deterministic natural-filename ordering. No random split is used.


In [13]:
# Cell 10 — S1GFloods PNG audit
# Reports the verified/required gray-domain statistics and saves CSVs under split_audit/.


def read_png_array(path: str) -> np.ndarray:
    with Image.open(path) as im:
        return np.asarray(im)


def read_png_gray(path: str) -> np.ndarray:
    arr = read_png_array(path)
    if arr.ndim == 2:
        return arr.astype(np.uint8)
    if arr.ndim == 3 and arr.shape[2] >= 1:
        return arr[..., 0].astype(np.uint8)
    raise ValueError(f"Unsupported PNG shape {arr.shape} for {path}")


def channels_identical(arr: np.ndarray) -> bool:
    if arr.ndim == 2:
        return True
    if arr.ndim != 3 or arr.shape[2] < 3:
        return False
    return bool(np.array_equal(arr[..., 0], arr[..., 1]) and np.array_equal(arr[..., 0], arr[..., 2]))


def _longest_true_run(v: np.ndarray) -> int:
    v = np.asarray(v, dtype=np.uint8)
    if not v.any():
        return 0
    padded = np.pad(v, (1, 1), constant_values=0)
    d = np.diff(padded.astype(np.int16))
    starts = np.where(d == 1)[0]
    ends = np.where(d == -1)[0]
    return int(np.max(ends - starts)) if len(starts) else 0


def both_zero_border_audit(a: np.ndarray, b: np.ndarray) -> Dict[str, Any]:
    z = (a == 0) & (b == 0)
    if not z.any():
        return {"both_zero_fraction": 0.0, "touches_border": False, "straight_edge_like": False,
                "interior_blob_like": False, "n_zero_components": 0}
    labeled, ncomp = cc_label(z.astype(np.uint8))
    straight = False
    interior_blob = False
    for cid in range(1, ncomp + 1):
        comp = labeled == cid
        touches = bool(comp[0, :].any() or comp[-1, :].any() or comp[:, 0].any() or comp[:, -1].any())
        if touches:
            runs = [_longest_true_run(comp[0, :]), _longest_true_run(comp[-1, :]),
                    _longest_true_run(comp[:, 0]), _longest_true_run(comp[:, -1])]
            if max(runs) >= max(16, int(0.06 * max(comp.shape))):
                straight = True
        elif int(comp.sum()) >= 4:
            interior_blob = True
    touches_border = bool(z[0, :].any() or z[-1, :].any() or z[:, 0].any() or z[:, -1].any())
    return {"both_zero_fraction": float(z.mean()), "touches_border": touches_border,
            "straight_edge_like": straight, "interior_blob_like": interior_blob,
            "n_zero_components": int(ncomp)}


def hist_median(hist: np.ndarray) -> float:
    total = hist.sum()
    if total == 0:
        return float("nan")
    return float(np.searchsorted(np.cumsum(hist), (total + 1) / 2))


def density_crossover(flood_hist: np.ndarray, land_hist: np.ndarray) -> float:
    if flood_hist.sum() == 0 or land_hist.sum() == 0:
        return float("nan")
    pf = gaussian_filter1d(flood_hist.astype(np.float64) / flood_hist.sum(), sigma=1.5)
    pl = gaussian_filter1d(land_hist.astype(np.float64) / land_hist.sum(), sigma=1.5)
    diff = pf - pl
    crossings = np.where(np.signbit(diff[:-1]) != np.signbit(diff[1:]))[0]
    if len(crossings):
        scores = (pf[crossings] + pl[crossings])
        return float(crossings[int(np.argmax(scores))] + 0.5)
    support = (pf + pl) > 1e-5
    idxs = np.where(support)[0]
    return float(idxs[np.argmin(np.abs(diff[idxs]))]) if len(idxs) else float("nan")

summary_rows = []
train_patch_rows = []
nodata_rows = []
hist_A = np.zeros(256, dtype=np.int64)
hist_B = np.zeros(256, dtype=np.int64)
hist_A_flood = np.zeros(256, dtype=np.int64)
hist_A_land = np.zeros(256, dtype=np.int64)
hist_B_flood = np.zeros(256, dtype=np.int64)
hist_B_land = np.zeros(256, dtype=np.int64)
train_pos = train_total = 0

for split in VALID_SPLITS:
    for t in tqdm(SPLITS[split], desc=f"Auditing {split} PNGs"):
        a_raw = read_png_array(t["pre"])
        b_raw = read_png_array(t["post"])
        gt_raw = read_png_array(t["label"])
        a = a_raw[..., 0] if a_raw.ndim == 3 else a_raw
        b = b_raw[..., 0] if b_raw.ndim == 3 else b_raw
        gt = gt_raw[..., 0] if gt_raw.ndim == 3 else gt_raw
        a = a.astype(np.uint8); b = b.astype(np.uint8); gt = gt.astype(np.uint8)

        shape_match = a.shape == b.shape == gt.shape
        if not shape_match:
            raise ValueError(f"Shape mismatch for {t['key']}: A={a.shape}, B={b.shape}, GT={gt.shape}")
        gt_values = np.unique(gt)
        if not set(gt_values.tolist()).issubset({0, 255}):
            raise ValueError(f"Unexpected GT values for {t['key']}: {gt_values.tolist()} (expected only 0/255)")

        za = float(np.mean(a == 0)); z255a = float(np.mean(a == 255))
        zb = float(np.mean(b == 0)); z255b = float(np.mean(b == 255))
        zero_audit = both_zero_border_audit(a, b)
        nodata_rows.append({"split": split, "key": t["key"], **zero_audit})
        summary_rows.append({
            "split": split, "key": t["key"], "shape": str(a.shape),
            "a_channels_identical": channels_identical(a_raw),
            "b_channels_identical": channels_identical(b_raw),
            "a_frac_0": za, "a_frac_255": z255a, "b_frac_0": zb, "b_frac_255": z255b,
            "gt_positive_fraction": float(np.mean(gt > 127)),
        })

        if split == "train":
            flood = gt > 127
            land = ~flood
            hist_A += np.bincount(a.ravel(), minlength=256)
            hist_B += np.bincount(b.ravel(), minlength=256)
            hist_A_flood += np.bincount(a[flood], minlength=256) if flood.any() else 0
            hist_A_land += np.bincount(a[land], minlength=256) if land.any() else 0
            hist_B_flood += np.bincount(b[flood], minlength=256) if flood.any() else 0
            hist_B_land += np.bincount(b[land], minlength=256) if land.any() else 0
            train_pos += int(flood.sum()); train_total += int(gt.size)
            land_delta = (b.astype(np.float32) - a.astype(np.float32))[land]
            train_patch_rows.append({
                "key": t["key"],
                "a_patch_median": float(np.median(a)),
                "b_patch_median": float(np.median(b)),
                "land_b_minus_a_median": float(np.median(land_delta)) if land_delta.size else np.nan,
                "gt_positive_fraction": float(flood.mean()),
            })

summary_df = pd.DataFrame(summary_rows)
train_patch_df = pd.DataFrame(train_patch_rows)
nodata_df = pd.DataFrame(nodata_rows)
summary_df.to_csv(os.path.join(DIRS["split_audit"], "png_dataset_summary.csv"), index=False)
train_patch_df.to_csv(os.path.join(DIRS["split_audit"], "train_patch_statistics.csv"), index=False)
nodata_df.to_csv(os.path.join(DIRS["split_audit"], "both_zero_border_audit.csv"), index=False)

TRAIN_POSITIVE_FRACTION = train_pos / max(train_total, 1)
TRAIN_GRAY_HISTS = {"A": hist_A, "B": hist_B}
TRAIN_CLASS_HISTS = {"A_flood": hist_A_flood, "A_land": hist_A_land, "B_flood": hist_B_flood, "B_land": hist_B_land}
TRAIN_GRAY_MEDIANS = {
    "A_flood": hist_median(hist_A_flood), "A_land": hist_median(hist_A_land),
    "B_flood": hist_median(hist_B_flood), "B_land": hist_median(hist_B_land),
}
FLOOD_NONFLOOD_CROSSOVER_GRAY = density_crossover(hist_B_flood, hist_B_land)

hist_df = pd.DataFrame({
    "gray": np.arange(256), "A_all": hist_A, "B_all": hist_B,
    "A_flood": hist_A_flood, "A_land": hist_A_land,
    "B_flood": hist_B_flood, "B_land": hist_B_land,
})
hist_df.to_csv(os.path.join(DIRS["split_audit"], "train_gray_histograms.csv"), index=False)

print("A/B channel identity fractions:",
      summary_df[["a_channels_identical", "b_channels_identical"]].mean().to_dict())
print("Train A: frac gray==0 / 255 =",
      float(hist_A[0] / hist_A.sum()), float(hist_A[255] / hist_A.sum()))
print("Train B: frac gray==0 / 255 =",
      float(hist_B[0] / hist_B.sum()), float(hist_B[255] / hist_B.sum()))
print(f"Train GT positive fraction: {TRAIN_POSITIVE_FRACTION:.4%}")
print("Class-conditional gray medians:", TRAIN_GRAY_MEDIANS)
print("Per-patch land (B-A) median distribution:")
print(train_patch_df["land_b_minus_a_median"].describe(percentiles=[0.25, 0.5, 0.75]).to_string())
print(f"Flood/non-flood B-gray density crossover ≈ {FLOOD_NONFLOOD_CROSSOVER_GRAY:.1f}")
print("Patch-level A/B median distribution:")
print(train_patch_df[["a_patch_median", "b_patch_median"]].describe().to_string())
print("Both-zero NoData-border heuristic fractions:", {
    "straight_edge_like": float(nodata_df["straight_edge_like"].mean()),
    "interior_blob_like": float(nodata_df["interior_blob_like"].mean()),
    "touches_border": float(nodata_df["touches_border"].mean()),
})
print("NOTE: the border/straight-edge classification is a heuristic audit, not a verified S1GFloods NoData annotation.")
if TRAIN_POSITIVE_FRACTION < 0.10:
    print("RECOMMENDATION ONLY: positive pixels are <10%; consider LOSS_NAME='focal_tversky' or 'bce_tversky'. "
          "The notebook does NOT change LOSS_NAME automatically.")

# Example A/B/GT panel
examples = SPLITS["train"][:4]
fig, axes = plt.subplots(len(examples), 3, figsize=(12, 4 * len(examples)))
if len(examples) == 1:
    axes = axes[None, :]
for i, t in enumerate(examples):
    a = read_png_gray(t["pre"]); b = read_png_gray(t["post"]); gt = read_png_gray(t["label"])
    for ax, img, title in zip(axes[i], [a, b, gt], ["A / PRE (VV gray)", "B / POST (VV gray)", "GT (new inundation)"]):
        ax.imshow(img, cmap="gray"); ax.set_title(f"{t['key']}\n{title}" if title.startswith("A") else title); ax.axis("off")
fig.tight_layout()
fig.savefig(os.path.join(DIRS["figures"], "s1gfloods_abgt_examples.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

Auditing train PNGs:   0%|          | 0/4300 [00:00<?, ?it/s]

Auditing val PNGs:   0%|          | 0/530 [00:00<?, ?it/s]

Auditing test PNGs:   0%|          | 0/530 [00:00<?, ?it/s]

A/B channel identity fractions: {'a_channels_identical': 1.0, 'b_channels_identical': 1.0}
Train A: frac gray==0 / 255 = 0.06602232112440952 0.02643187766851381
Train B: frac gray==0 / 255 = 0.09278441318245821 0.018248862333075946
Train GT positive fraction: 33.9074%
Class-conditional gray medians: {'A_flood': 136.0, 'A_land': 144.0, 'B_flood': 39.0, 'B_land': 173.0}
Per-patch land (B-A) median distribution:
count    4300.000000
mean       22.892791
std        21.443606
min       -70.000000
25%         8.000000
50%        24.000000
75%        36.000000
max       113.000000
Flood/non-flood B-gray density crossover ≈ 98.5
Patch-level A/B median distribution:
       a_patch_median  b_patch_median
count     4300.000000     4300.000000
mean       133.397674      118.134186
std         38.113132       65.370659
min          0.000000        0.000000
25%        120.000000       63.000000
50%        136.000000      137.000000
75%        160.000000      176.000000
max        240.000000      237

In [14]:
# Cell 11 — S1GFloods label-semantics audit / documentation
# VERIFIED for this notebook: GT=255 is newly inundated flood; permanent water is negative (0).
# Therefore PRE/POST order is semantically asymmetric and must NEVER be swapped as augmentation.

semantics_rows = [{
    "target": "GT > 127",
    "positive_semantics": "newly inundated flood",
    "permanent_water": "negative / GT=0",
    "temporal_order": "A=PRE, B=POST; asymmetric; no date swapping",
    "train_positive_fraction": TRAIN_POSITIVE_FRACTION,
    "median_A_flood": TRAIN_GRAY_MEDIANS["A_flood"],
    "median_A_land": TRAIN_GRAY_MEDIANS["A_land"],
    "median_B_flood": TRAIN_GRAY_MEDIANS["B_flood"],
    "median_B_land": TRAIN_GRAY_MEDIANS["B_land"],
    "density_crossover_B_gray": FLOOD_NONFLOOD_CROSSOVER_GRAY,
}]
semantics_df = pd.DataFrame(semantics_rows)
semantics_df.to_csv(os.path.join(DIRS["split_audit"], "label_semantics_audit.csv"), index=False)
print(semantics_df.to_string(index=False))

examples = SPLITS["train"][:4]
fig, axes = plt.subplots(len(examples), 4, figsize=(16, 4 * len(examples)))
if len(examples) == 1:
    axes = axes[None, :]
for i, t in enumerate(examples):
    a = read_png_gray(t["pre"]); b = read_png_gray(t["post"]); gt = read_png_gray(t["label"])
    diff = b.astype(np.float32) - a.astype(np.float32)
    panels = [(a, "A / PRE"), (b, "B / POST"), (diff, "B-A gray difference"), (gt > 127, "GT: new inundation")]
    for ax, (img, title) in zip(axes[i], panels):
        ax.imshow(img, cmap="gray" if "difference" not in title else "RdBu", vmin=(-80 if "difference" in title else None), vmax=(80 if "difference" in title else None))
        ax.set_title(f"{t['key']}\n{title}" if title == "A / PRE" else title); ax.axis("off")
fig.tight_layout()
fig.savefig(os.path.join(DIRS["figures"], "s1gfloods_label_semantics_examples.png"), dpi=200, bbox_inches="tight")
plt.close(fig)
print("Interpretation: train/evaluate as newly inundated flood change. Do not swap A and B.")

  target    positive_semantics permanent_water                              temporal_order  train_positive_fraction  median_A_flood  median_A_land  median_B_flood  median_B_land  density_crossover_B_gray
GT > 127 newly inundated flood negative / GT=0 A=PRE, B=POST; asymmetric; no date swapping                 0.339074           136.0          144.0            39.0          173.0                      98.5
Interpretation: train/evaluate as newly inundated flood change. Do not swap A and B.


In [15]:
# Cell 12 — S1GFloods preprocessing: SINGLE source of truth for PNG training and GeoTIFF inference

class S1GFloodsPreprocessor:
    def __init__(self, lo: float, hi: float, quantize: bool, in_channels: int, input_mode: str,
                 png_scale: float = 255.0):
        if not hi > lo:
            raise ValueError(f"Expected hi > lo, got lo={lo}, hi={hi}")
        if input_mode not in {"1ch", "3ch_replicated"}:
            raise ValueError(f"Unsupported input_mode={input_mode}")
        expected_channels = 1 if input_mode == "1ch" else 3
        if int(in_channels) != expected_channels:
            raise ValueError(f"in_channels={in_channels} inconsistent with input_mode={input_mode}")
        self.lo = float(lo); self.hi = float(hi); self.quantize = bool(quantize)
        self.in_channels = int(in_channels); self.input_mode = input_mode; self.png_scale = float(png_scale)

    def from_png_gray(self, g_uint8: np.ndarray) -> np.ndarray:
        g = np.asarray(g_uint8, dtype=np.float32)
        return np.clip(g / self.png_scale, 0.0, 1.0).astype(np.float32)

    def from_db(self, vv_db: np.ndarray, fill_gray: float = 0.0) -> Tuple[np.ndarray, np.ndarray]:
        vv = np.asarray(vv_db, dtype=np.float32)
        valid = np.isfinite(vv)
        safe = np.where(valid, vv, self.lo).astype(np.float32)
        x = np.clip((safe - self.lo) / (self.hi - self.lo), 0.0, 1.0)
        if self.quantize:
            x = np.round(x * self.png_scale) / self.png_scale
        fill_norm = float(np.clip(fill_gray / self.png_scale, 0.0, 1.0))
        x[~valid] = fill_norm
        return x.astype(np.float32), valid

    def expand_channels(self, x01: np.ndarray) -> np.ndarray:
        x = np.asarray(x01, dtype=np.float32)
        if x.ndim != 2:
            raise ValueError(f"Expected 2D gray array, got {x.shape}")
        if self.input_mode == "1ch":
            return x[None, ...]
        return np.repeat(x[None, ...], 3, axis=0)

    def png_to_model(self, g_uint8: np.ndarray) -> np.ndarray:
        return self.expand_channels(self.from_png_gray(g_uint8))

    def db_to_model(self, vv_db: np.ndarray, fill_gray: float = 0.0) -> Tuple[np.ndarray, np.ndarray]:
        x, valid = self.from_db(vv_db, fill_gray=fill_gray)
        return self.expand_channels(x), valid

    def to_dict(self) -> Dict[str, Any]:
        return {
            "lo": self.lo, "hi": self.hi, "quantize": self.quantize,
            "in_channels": self.in_channels, "input_mode": self.input_mode,
            "png_scale": self.png_scale, "expected_units": "dB", "band_name": "VV",
            "expected_resolution_m": 10,
        }

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "S1GFloodsPreprocessor":
        return cls(d["lo"], d["hi"], d["quantize"], d["in_channels"], d["input_mode"], d.get("png_scale", 255.0))

PREPROCESSOR = S1GFloodsPreprocessor(PNG_DB_LO, PNG_DB_HI, PNG_QUANTIZE, IN_CHANNELS, INPUT_MODE)
print("Preprocessor:", PREPROCESSOR.to_dict())
print("UNVERIFIED: checkpoint will preserve the inferred dB→gray range", (PNG_DB_LO, PNG_DB_HI))
with open(os.path.join(DIRS["configs"], "preprocessing.json"), "w") as f:
    json.dump(PREPROCESSOR.to_dict(), f, indent=2)

Preprocessor: {'lo': -30.0, 'hi': 0.0, 'quantize': True, 'in_channels': 1, 'input_mode': '1ch', 'png_scale': 255.0, 'expected_units': 'dB', 'band_name': 'VV', 'expected_resolution_m': 10}
UNVERIFIED: checkpoint will preserve the inferred dB→gray range (-30.0, 0.0)


In [16]:
# Cell 13 — Synchronized spatial + gray-domain radiometric augmentation

class SyncedSpatialAugment:
    """Apply one sampled flip/rotation identically to PRE, POST, label, and valid mask."""
    def __init__(self, enabled: bool = True):
        self.enabled = enabled

    def __call__(self, pre: np.ndarray, post: np.ndarray, label: np.ndarray, valid_mask: np.ndarray):
        if not self.enabled:
            return pre, post, label, valid_mask
        hflip = random.random() < 0.5
        vflip = random.random() < 0.5
        k = random.choice([0, 1, 2, 3])

        arrays = [pre, post, label, valid_mask]
        out = []
        for arr in arrays:
            x = arr
            if hflip: x = x[:, ::-1]
            if vflip: x = x[::-1, :]
            if k: x = np.rot90(x, k, axes=(0, 1))
            out.append(x.copy())
        return tuple(out)


class SARRadiometricAugment:
    """Training-only augmentation in S1GFloods 8-bit GRAY units. PRE/POST stay paired and are never swapped."""
    def __init__(self, enabled=True, dark_tail=True, gain_offset=True, date_offset=True, blur=True, quantize=True):
        self.enabled = enabled
        self.dark_tail = dark_tail; self.gain_offset = gain_offset
        self.date_offset = date_offset; self.blur = blur; self.quantize = quantize

    def __call__(self, pre: np.ndarray, post: np.ndarray):
        if not self.enabled:
            return pre, post
        pre = pre.astype(np.float32, copy=True); post = post.astype(np.float32, copy=True)

        # (a) Joint dark-tail compression, p=0.5: simulate external scenes with a weak/missing dark tail.
        if self.dark_tail and random.random() < 0.5:
            if random.random() < 0.5:
                f = random.uniform(0.0, 60.0)
                pre = f + pre * (255.0 - f) / 255.0
                post = f + post * (255.0 - f) / 255.0
            else:
                f = random.uniform(0.0, 85.0)
                pre = np.maximum(pre, f); post = np.maximum(post, f)

        # (b) Joint gain and offset.
        if self.gain_offset:
            gain = random.uniform(0.85, 1.15)
            offset = random.uniform(-15.0, 15.0)
            pre = pre * gain + offset; post = post * gain + offset

        # (c) Independent small per-date offsets, p=0.5.
        if self.date_offset and random.random() < 0.5:
            pre = pre + np.random.normal(0.0, 8.0)
            post = post + np.random.normal(0.0, 8.0)

        # (d) Joint blur strength (same sigma, separately applied to each date), p=0.3.
        if self.blur and random.random() < 0.3:
            sigma = random.uniform(0.5, 1.0)
            pre = gaussian_filter(pre, sigma=sigma, mode="reflect")
            post = gaussian_filter(post, sigma=sigma, mode="reflect")

        # (e) Return to legal S1GFloods gray domain.
        pre = np.clip(pre, 0.0, 255.0); post = np.clip(post, 0.0, 255.0)
        if self.quantize:
            pre = np.round(pre); post = np.round(post)
        return pre.astype(np.float32), post.astype(np.float32)

SPATIAL_AUG = SyncedSpatialAugment(enabled=USE_AUGMENTATION)
RADIOMETRIC_AUG = SARRadiometricAugment(
    enabled=USE_SAR_RADIOMETRIC_AUG,
    dark_tail=AUG_DARK_TAIL, gain_offset=AUG_GAIN_OFFSET,
    date_offset=AUG_DATE_OFFSET, blur=AUG_BLUR, quantize=PNG_QUANTIZE,
)
print("Spatial augmentation:", USE_AUGMENTATION)
print("Radiometric augmentation:", USE_SAR_RADIOMETRIC_AUG,
      {"dark_tail": AUG_DARK_TAIL, "gain_offset": AUG_GAIN_OFFSET, "date_offset": AUG_DATE_OFFSET, "blur": AUG_BLUR})

Spatial augmentation: True
Radiometric augmentation: True {'dark_tail': True, 'gain_offset': True, 'date_offset': True, 'blur': True}


In [17]:
# Cell 14 — PyTorch Dataset for S1GFloods PNG A/B/GT

class S1GFloodsChangeDataset(Dataset):
    def __init__(self, triplets: List[Dict[str, str]], preprocessor: S1GFloodsPreprocessor,
                 spatial_aug: Optional[SyncedSpatialAugment] = None,
                 radiometric_aug: Optional[SARRadiometricAugment] = None):
        self.triplets = triplets
        self.preprocessor = preprocessor
        self.spatial_aug = spatial_aug
        self.radiometric_aug = radiometric_aug

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        t = self.triplets[idx]
        pre_gray = read_png_gray(t["pre"]).astype(np.float32)
        post_gray = read_png_gray(t["post"]).astype(np.float32)
        gt = read_png_gray(t["label"])
        assert pre_gray.shape == post_gray.shape == gt.shape, (
            f"Shape mismatch for {t['key']}: pre={pre_gray.shape}, post={post_gray.shape}, GT={gt.shape}"
        )

        label = (gt > 127).astype(np.float32)
        valid_mask = np.ones_like(label, dtype=np.bool_)
        if MASK_BOTH_ZERO:
            valid_mask &= ~((pre_gray == 0) & (post_gray == 0))

        # Keep GRAY [0,255] through radiometric augmentation; normalize only at the end.
        if self.radiometric_aug is not None:
            pre_gray, post_gray = self.radiometric_aug(pre_gray, post_gray)
        if self.spatial_aug is not None:
            pre_gray, post_gray, label, valid_mask = self.spatial_aug(pre_gray, post_gray, label, valid_mask)

        pre = self.preprocessor.expand_channels(self.preprocessor.from_png_gray(pre_gray))
        post = self.preprocessor.expand_channels(self.preprocessor.from_png_gray(post_gray))
        pre = np.nan_to_num(pre, nan=0.0, posinf=1.0, neginf=0.0).astype(np.float32)
        post = np.nan_to_num(post, nan=0.0, posinf=1.0, neginf=0.0).astype(np.float32)

        return {
            "pre": torch.from_numpy(pre).float(),
            "post": torch.from_numpy(post).float(),
            "target": torch.from_numpy(label.astype(np.float32)),
            "valid_mask": torch.from_numpy(valid_mask.astype(np.bool_)),
            "key": t["key"],
        }

In [18]:
# Cell 15 — DataLoaders

TRAIN_DS = S1GFloodsChangeDataset(SPLITS["train"], PREPROCESSOR, SPATIAL_AUG, RADIOMETRIC_AUG)
VAL_DS   = S1GFloodsChangeDataset(SPLITS["val"], PREPROCESSOR, spatial_aug=None, radiometric_aug=None)
TEST_DS  = S1GFloodsChangeDataset(SPLITS["test"], PREPROCESSOR, spatial_aug=None, radiometric_aug=None)

def make_loader(ds, batch_size, shuffle, sampler=None):
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=(shuffle and sampler is None),
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        persistent_workers=False
    )

TRAIN_LOADER = make_loader(TRAIN_DS, BATCH_SIZE_PER_GPU, shuffle=True)
VAL_LOADER = make_loader(VAL_DS, BATCH_SIZE_PER_GPU, shuffle=False)
TEST_LOADER = make_loader(TEST_DS, BATCH_SIZE_PER_GPU, shuffle=False)

print(f"Train batches: {len(TRAIN_LOADER)} | Val batches: {len(VAL_LOADER)} | Test batches: {len(TEST_LOADER)}")
batch = next(iter(TRAIN_LOADER))
print("Sample batch shapes -> pre:", batch["pre"].shape, "post:", batch["post"].shape,
      "target:", batch["target"].shape, "valid_mask:", batch["valid_mask"].shape)
print(f"Configured micro-batch/GPU={BATCH_SIZE_PER_GPU}, ACCUM_STEPS={ACCUM_STEPS}. "
      "Effective global batch also depends on the number of active training processes/GPUs.")

Train batches: 1075 | Val batches: 133 | Test batches: 133
Sample batch shapes -> pre: torch.Size([4, 1, 256, 256]) post: torch.Size([4, 1, 256, 256]) target: torch.Size([4, 256, 256]) valid_mask: torch.Size([4, 256, 256])
Configured micro-batch/GPU=4, ACCUM_STEPS=4. Effective global batch also depends on the number of active training processes/GPUs.


In [19]:
# Cell 16 — Model zoo + factory. All model-specific logic lives here only.

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class UNetEncoder(nn.Module):
    def __init__(self, in_ch, base=32):
        super().__init__()
        chs = [base, base*2, base*4, base*8, base*16]
        self.enc1 = ConvBlock(in_ch, chs[0]); self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(chs[0], chs[1]); self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(chs[1], chs[2]); self.pool3 = nn.MaxPool2d(2)
        self.enc4 = ConvBlock(chs[2], chs[3]); self.pool4 = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(chs[3], chs[4])
        self.chs = chs
    def forward(self, x):
        e1 = self.enc1(x); p1 = self.pool1(e1)
        e2 = self.enc2(p1); p2 = self.pool2(e2)
        e3 = self.enc3(p2); p3 = self.pool3(e3)
        e4 = self.enc4(p3); p4 = self.pool4(e4)
        b = self.bottleneck(p4)
        return b, [e1, e2, e3, e4]


class UNetDecoder(nn.Module):
    def __init__(self, chs, out_ch=1):
        super().__init__()
        self.up4 = nn.ConvTranspose2d(chs[4], chs[3], 2, stride=2); self.dec4 = ConvBlock(chs[3]*2, chs[3])
        self.up3 = nn.ConvTranspose2d(chs[3], chs[2], 2, stride=2); self.dec3 = ConvBlock(chs[2]*2, chs[2])
        self.up2 = nn.ConvTranspose2d(chs[2], chs[1], 2, stride=2); self.dec2 = ConvBlock(chs[1]*2, chs[1])
        self.up1 = nn.ConvTranspose2d(chs[1], chs[0], 2, stride=2); self.dec1 = ConvBlock(chs[0]*2, chs[0])
        self.out_conv = nn.Conv2d(chs[0], out_ch, 1)
    def forward(self, b, skips):
        e1, e2, e3, e4 = skips
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)  # raw logits


class BiTemporalUNet(nn.Module):
    """Early-fusion baseline: concatenates PRE+POST (2*IN_CHANNELS) into a single U-Net."""
    def __init__(self, in_ch=IN_CHANNELS, base=32):
        super().__init__()
        self.encoder = UNetEncoder(in_ch=2 * in_ch, base=base)
        self.decoder = UNetDecoder(self.encoder.chs, out_ch=1)
    def forward(self, pre, post):
        x = torch.cat([pre, post], dim=1)
        b, skips = self.encoder(x)
        return self.decoder(b, skips)  # logits [B,1,H,W]


class SiameseUNetEncoder(nn.Module):
    """Weight-shared encoder applied to PRE and POST separately (Siamese)."""
    def __init__(self, in_ch=IN_CHANNELS, base=32):
        super().__init__()
        self.shared = UNetEncoder(in_ch=in_ch, base=base)
    def forward(self, pre, post):
        b_pre, s_pre = self.shared(pre)
        b_post, s_post = self.shared(post)
        return (b_pre, b_post), (s_pre, s_post)


class SiameseUNet(nn.Module):
    """Siamese U-Net with feature concatenation at bottleneck+skips (FC-Siam-conc style)."""
    def __init__(self, in_ch=IN_CHANNELS, base=32):
        super().__init__()
        self.encoder = SiameseUNetEncoder(in_ch=in_ch, base=base)
        chs = self.encoder.shared.chs
        cat_chs = [c*2 for c in chs]
        self.decoder = UNetDecoder(cat_chs, out_ch=1)
    def forward(self, pre, post):
        (b_pre, b_post), (s_pre, s_post) = self.encoder(pre, post)
        b = torch.cat([b_pre, b_post], dim=1)
        skips = [torch.cat([a, c], dim=1) for a, c in zip(s_pre, s_post)]
        return self.decoder(b, skips)


class FCSiamDiff(nn.Module):
    """FC-Siam-diff: Siamese encoder, decoder consumes |feat_pre - feat_post| (Daudt et al. style)."""
    def __init__(self, in_ch=IN_CHANNELS, base=32):
        super().__init__()
        self.encoder = SiameseUNetEncoder(in_ch=in_ch, base=base)
        chs = self.encoder.shared.chs
        self.decoder = UNetDecoder(chs, out_ch=1)
    def forward(self, pre, post):
        (b_pre, b_post), (s_pre, s_post) = self.encoder(pre, post)
        b = torch.abs(b_pre - b_post)
        skips = [torch.abs(a - c) for a, c in zip(s_pre, s_post)]
        return self.decoder(b, skips)


class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        psi = self.psi(self.relu(self.W_g(g) + self.W_x(x)))
        return x * psi


class AttentionUNet(nn.Module):
    """Early-fusion input with attention gates on the skip connections."""
    def __init__(self, in_ch=IN_CHANNELS, base=32):
        super().__init__()
        self.encoder = UNetEncoder(in_ch=2 * in_ch, base=base)
        chs = self.encoder.chs
        self.att4 = AttentionGate(chs[3], chs[3], chs[3]//2)
        self.att3 = AttentionGate(chs[2], chs[2], chs[2]//2)
        self.att2 = AttentionGate(chs[1], chs[1], chs[1]//2)
        self.att1 = AttentionGate(chs[0], chs[0], chs[0]//2)
        self.up4 = nn.ConvTranspose2d(chs[4], chs[3], 2, stride=2); self.dec4 = ConvBlock(chs[3]*2, chs[3])
        self.up3 = nn.ConvTranspose2d(chs[3], chs[2], 2, stride=2); self.dec3 = ConvBlock(chs[2]*2, chs[2])
        self.up2 = nn.ConvTranspose2d(chs[2], chs[1], 2, stride=2); self.dec2 = ConvBlock(chs[1]*2, chs[1])
        self.up1 = nn.ConvTranspose2d(chs[1], chs[0], 2, stride=2); self.dec1 = ConvBlock(chs[0]*2, chs[0])
        self.out_conv = nn.Conv2d(chs[0], 1, 1)
    def forward(self, pre, post):
        x = torch.cat([pre, post], dim=1)
        b, (e1, e2, e3, e4) = self.encoder(x)
        g4 = self.up4(b); e4a = self.att4(g4, e4); d4 = self.dec4(torch.cat([g4, e4a], dim=1))
        g3 = self.up3(d4); e3a = self.att3(g3, e3); d3 = self.dec3(torch.cat([g3, e3a], dim=1))
        g2 = self.up2(d3); e2a = self.att2(g2, e2); d2 = self.dec2(torch.cat([g2, e2a], dim=1))
        g1 = self.up1(d2); e1a = self.att1(g1, e1); d1 = self.dec1(torch.cat([g1, e1a], dim=1))
        return self.out_conv(d1)


# =====================================================================================
# BIT — Bitemporal Image Transformer (Chen, Qi & Shi, 2021)
# Shared lightweight CNN backbone -> a handful of learned "semantic tokens" per image ->
# a joint transformer encoder over the concatenated pre+post tokens (context exchange) ->
# a per-image transformer decoder where every pixel attends back to the refined tokens ->
# absolute feature difference -> conv head -> upsample to full resolution.
# Adapted for 2-channel (VV, VH) SAR input instead of 3-channel optical.
# =====================================================================================

class BasicResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = None
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm2d(out_ch))
    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class BITBackbone(nn.Module):
    """Lightweight ResNet-style stem, downsamples to 1/8 resolution (no ImageNet weights — SAR input)."""
    def __init__(self, in_ch=IN_CHANNELS, out_ch=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 32, 7, stride=2, padding=3), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )  # 1/4
        self.layer1 = self._make_layer(32, 32, 2, stride=1)      # 1/4
        self.layer2 = self._make_layer(32, 64, 2, stride=2)      # 1/8
        self.layer3 = self._make_layer(64, out_ch, 2, stride=1)  # 1/8, channel-reduced for the tokenizer
    def _make_layer(self, in_ch, out_ch, blocks, stride):
        layers = [BasicResBlock(in_ch, out_ch, stride)]
        for _ in range(blocks - 1):
            layers.append(BasicResBlock(out_ch, out_ch, 1))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.stem(x); x = self.layer1(x); x = self.layer2(x); x = self.layer3(x)
        return x  # [B, out_ch, H/8, W/8]


class SemanticTokenizer(nn.Module):
    """Pools the spatial feature map into L learned 'semantic tokens' via spatial attention."""
    def __init__(self, in_ch, num_tokens):
        super().__init__()
        self.attn = nn.Conv2d(in_ch, num_tokens, 1)
    def forward(self, x):
        a = self.attn(x)                                   # [B, L, H, W]
        a = rearrange(a, 'b l h w -> b l (h w)')
        a = torch.softmax(a, dim=-1)
        feat = rearrange(x, 'b c h w -> b (h w) c')
        return torch.bmm(a, feat)                           # [B, L, C]


class CrossAttentionBlock(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim); self.norm_kv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim*2), nn.GELU(), nn.Linear(dim*2, dim))
    def forward(self, q, kv):
        attn_out, _ = self.attn(self.norm_q(q), self.norm_kv(kv), self.norm_kv(kv))
        q = q + attn_out
        return q + self.mlp(self.norm2(q))


class TokenTransformerDecoder(nn.Module):
    """Every pixel (as a query) attends back to the refined context tokens (as keys/values)."""
    def __init__(self, dim, heads=8, depth=8):
        super().__init__()
        self.layers = nn.ModuleList([CrossAttentionBlock(dim, heads) for _ in range(depth)])
    def forward(self, x, tokens):
        B, C, H, W = x.shape
        q = rearrange(x, 'b c h w -> b (h w) c')
        for layer in self.layers:
            q = layer(q, tokens)
        return rearrange(q, 'b (h w) c -> b c h w', h=H, w=W)


class BIT(nn.Module):
    def __init__(self, in_ch=IN_CHANNELS, backbone_dim=32, num_tokens=4, enc_depth=1, dec_depth=8, heads=8, base=32):
        super().__init__()
        self.backbone = BITBackbone(in_ch=in_ch, out_ch=backbone_dim)
        self.tokenizer = SemanticTokenizer(backbone_dim, num_tokens)
        enc_layer = nn.TransformerEncoderLayer(d_model=backbone_dim, nhead=heads,
                                                dim_feedforward=backbone_dim*2, batch_first=True, activation="gelu")
        self.token_encoder = nn.TransformerEncoder(enc_layer, num_layers=enc_depth)
        self.decoder = TokenTransformerDecoder(backbone_dim, heads=heads, depth=dec_depth)
        self.head = nn.Sequential(
            nn.Conv2d(backbone_dim, base, 3, padding=1), nn.BatchNorm2d(base), nn.ReLU(inplace=True),
            nn.Conv2d(base, base, 3, padding=1), nn.BatchNorm2d(base), nn.ReLU(inplace=True),
            nn.Conv2d(base, 1, 1),
        )
    def forward(self, pre, post):
        H, W = pre.shape[-2:]
        f_pre, f_post = self.backbone(pre), self.backbone(post)
        t_pre, t_post = self.tokenizer(f_pre), self.tokenizer(f_post)
        tokens = self.token_encoder(torch.cat([t_pre, t_post], dim=1))
        L = t_pre.shape[1]
        t_pre, t_post = tokens[:, :L], tokens[:, L:]
        f_pre, f_post = self.decoder(f_pre, t_pre), self.decoder(f_post, t_post)
        out = self.head(torch.abs(f_pre - f_post))
        return F.interpolate(out, size=(H, W), mode="bilinear", align_corners=False)


# =====================================================================================
# ChangeFormer (Bandara & Patel, 2022)
# Siamese hierarchical Mix-Transformer (MiT/SegFormer-style) encoder with 4 stages of
# overlap-patch embedding + spatial-reduction self-attention + Mix-FFN -> per-scale
# |feat_pre - feat_post| -> SegFormer-style all-MLP decoder fusing the 4 scales -> upsample.
# Adapted for 2-channel (VV, VH) SAR input instead of 3-channel optical.
# =====================================================================================

# class OverlapPatchEmbed(nn.Module):
#     def __init__(self, in_ch, embed_dim, patch_size, stride):
#         super().__init__()
#         self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=stride, padding=patch_size//2)
#         self.norm = nn.LayerNorm(embed_dim)
#     def forward(self, x):
#         x = self.proj(x)
#         _, _, H, W = x.shape
#         x = rearrange(x, 'b c h w -> b (h w) c')
#         return self.norm(x), H, W


# class EfficientSelfAttention(nn.Module):
#     """Spatial-reduction attention (SRA), as in PVT/SegFormer, to keep attention tractable at high resolution."""
#     def __init__(self, dim, heads, sr_ratio):
#         super().__init__()
#         self.heads = heads
#         self.scale = (dim // heads) ** -0.5
#         self.q = nn.Linear(dim, dim)
#         self.kv = nn.Linear(dim, dim*2)
#         self.proj = nn.Linear(dim, dim)
#         self.sr_ratio = sr_ratio
#         if sr_ratio > 1:
#             self.sr = nn.Conv2d(dim, dim, kernel_size=sr_ratio, stride=sr_ratio)
#             self.norm = nn.LayerNorm(dim)
#     def forward(self, x, H, W):
#         q = rearrange(self.q(x), 'b n (h d) -> b h n d', h=self.heads)
#         if self.sr_ratio > 1:
#             x_ = rearrange(x, 'b (h w) c -> b c h w', h=H, w=W)
#             x_ = rearrange(self.sr(x_), 'b c h w -> b (h w) c')
#             x_ = self.norm(x_)
#         else:
#             x_ = x
#         k, v = self.kv(x_).chunk(2, dim=-1)
#         k = rearrange(k, 'b n (h d) -> b h n d', h=self.heads)
#         v = rearrange(v, 'b n (h d) -> b h n d', h=self.heads)
#         attn = (q @ k.transpose(-2, -1) * self.scale).softmax(dim=-1)
#         out = rearrange(attn @ v, 'b h n d -> b n (h d)')
#         return self.proj(out)


# class MixFFN(nn.Module):
#     """FFN with a 3x3 depthwise conv for local position info (SegFormer's Mix-FFN)."""
#     def __init__(self, dim, hidden_dim):
#         super().__init__()
#         self.fc1 = nn.Linear(dim, hidden_dim)
#         self.dwconv = nn.Conv2d(hidden_dim, hidden_dim, 3, padding=1, groups=hidden_dim)
#         self.act = nn.GELU()
#         self.fc2 = nn.Linear(hidden_dim, dim)
#     def forward(self, x, H, W):
#         x = self.fc1(x)
#         x = rearrange(x, 'b (h w) c -> b c h w', h=H, w=W)
#         x = self.dwconv(x)
#         x = rearrange(x, 'b c h w -> b (h w) c')
#         return self.fc2(self.act(x))


# class MiTBlock(nn.Module):
#     def __init__(self, dim, heads, sr_ratio, mlp_ratio=4):
#         super().__init__()
#         self.norm1 = nn.LayerNorm(dim); self.attn = EfficientSelfAttention(dim, heads, sr_ratio)
#         self.norm2 = nn.LayerNorm(dim); self.ffn = MixFFN(dim, dim*mlp_ratio)
#     def forward(self, x, H, W):
#         x = x + self.attn(self.norm1(x), H, W)
#         return x + self.ffn(self.norm2(x), H, W)


# class MixTransformerEncoder(nn.Module):
#     """Simplified MiT (SegFormer) hierarchical Siamese encoder: 4 stages, decreasing resolution."""
#     def __init__(self, in_ch=IN_CHANNELS, embed_dims=(32,64,160,256), depths=(2,2,2,2), heads=(1,2,5,8), sr_ratios=(8,4,2,1)):
#         super().__init__()
#         patch_sizes, strides = (7, 3, 3, 3), (4, 2, 2, 2)
#         self.stages = nn.ModuleList()
#         prev_ch = in_ch
#         for i in range(4):
#             embed = OverlapPatchEmbed(prev_ch, embed_dims[i], patch_sizes[i], strides[i])
#             blocks = nn.ModuleList([MiTBlock(embed_dims[i], heads[i], sr_ratios[i]) for _ in range(depths[i])])
#             norm = nn.LayerNorm(embed_dims[i])
#             self.stages.append(nn.ModuleDict({"embed": embed, "blocks": blocks, "norm": norm}))
#             prev_ch = embed_dims[i]
#     def forward(self, x):
#         feats = []
#         for stage in self.stages:
#             x, H, W = stage["embed"](x)
#             for blk in stage["blocks"]:
#                 x = blk(x, H, W)
#             x = stage["norm"](x)
#             x = rearrange(x, 'b (h w) c -> b c h w', h=H, w=W)
#             feats.append(x)
#         return feats  # 4 feature maps at 1/4, 1/8, 1/16, 1/32 resolution


# class SegFormerMLPDecoder(nn.Module):
#     def __init__(self, embed_dims, decoder_dim=256, num_classes=1):
#         super().__init__()
#         self.linears = nn.ModuleList([nn.Conv2d(c, decoder_dim, 1) for c in embed_dims])
#         self.fuse = nn.Sequential(nn.Conv2d(decoder_dim*4, decoder_dim, 1), nn.BatchNorm2d(decoder_dim), nn.ReLU(inplace=True))
#         self.pred = nn.Conv2d(decoder_dim, num_classes, 1)
#     def forward(self, feats):
#         target_size = feats[0].shape[-2:]
#         ups = [F.interpolate(lin(f), size=target_size, mode="bilinear", align_corners=False)
#                for f, lin in zip(feats, self.linears)]
#         return self.pred(self.fuse(torch.cat(ups, dim=1)))


# class ChangeFormer(nn.Module):
#     def __init__(self, in_ch=IN_CHANNELS, embed_dims=(32,64,160,256), depths=(2,2,2,2), heads=(1,2,5,8), decoder_dim=256):
#         super().__init__()
#         self.encoder = MixTransformerEncoder(in_ch=in_ch, embed_dims=embed_dims, depths=depths, heads=heads)
#         self.decoder = SegFormerMLPDecoder(embed_dims, decoder_dim=decoder_dim, num_classes=1)
#     def forward(self, pre, post):
#         H, W = pre.shape[-2:]
#         feats_pre, feats_post = self.encoder(pre), self.encoder(post)
#         diff_feats = [torch.abs(a - b) for a, b in zip(feats_pre, feats_post)]
#         out = self.decoder(diff_feats)
#         return F.interpolate(out, size=(H, W), mode="bilinear", align_corners=False)

# # =====================================================================================
# # ChangeMamba (Chen, Song, Han, Xia & Yokoya, 2024) — imported from the cloned repo,
# # not reimplemented here (it depends on a compiled CUDA selective-scan kernel).
# # Wrapped so it exposes the same (pre, post) -> single-channel-logit interface as every
# # other model in MODEL_REGISTRY, so no other cell (loss/metrics/threshold/inference) changes.
# # =====================================================================================
# from changedetection.models.ChangeMambaBCD import ChangeMambaBCD  # noqa: E402

# class ChangeMambaWrapper(nn.Module):
#     def __init__(self, variant="tiny", encoder_pretrained=None, in_chans=IN_CHANNELS):
#         super().__init__()
#         depths, embed_dim = _CHANGEMAMBA_VARIANTS[variant]
#         self.net = ChangeMambaBCD(
#             pretrained=encoder_pretrained,
#             patch_size=4,
#             in_chans=in_chans,
#             num_classes=1000,
#             depths=depths,
#             dims=embed_dim,
#             ssm_d_state=1, ssm_ratio=2.0, ssm_rank_ratio=2.0, ssm_dt_rank="auto",
#             ssm_act_layer="silu", ssm_conv=3, ssm_conv_bias=False, ssm_drop_rate=0.0,
#             ssm_init="v0", forward_type="v3noz",
#             mlp_ratio=4.0, mlp_act_layer="gelu", mlp_drop_rate=0.0,
#             drop_path_rate=0.2, patch_norm=True, norm_layer="ln",
#             downsample_version="v3", patchembed_version="v2", gmlp=False,
#             use_checkpoint=False,   # <-- add this: gradient checkpointing for VSS blocks, not optional here
#         )

#     def forward(self, pre, post):
#         out = self.net(pre, post)
#         return (out[:, 1:2] - out[:, 0:1])


MODEL_REGISTRY = {
    "bitemporal_unet": lambda cfg: BiTemporalUNet(in_ch=cfg.get("IN_CHANNELS", IN_CHANNELS), base=cfg.get("unet_base", 32)),
    "siamese_unet":    lambda cfg: SiameseUNet(in_ch=cfg.get("IN_CHANNELS", IN_CHANNELS), base=cfg.get("unet_base", 32)),
    "fc_siam_conc":    lambda cfg: SiameseUNet(in_ch=cfg.get("IN_CHANNELS", IN_CHANNELS), base=cfg.get("unet_base", 32)),
    "fc_siam_diff":    lambda cfg: FCSiamDiff(in_ch=cfg.get("IN_CHANNELS", IN_CHANNELS), base=cfg.get("unet_base", 32)),
    "attention_unet":  lambda cfg: AttentionUNet(in_ch=cfg.get("IN_CHANNELS", IN_CHANNELS), base=cfg.get("unet_base", 32)),
    "bit": lambda cfg: BIT(
        in_ch=cfg.get("IN_CHANNELS", IN_CHANNELS), backbone_dim=cfg.get("BIT_DIM", 32), num_tokens=cfg.get("BIT_TOKENS", 4),
        enc_depth=cfg.get("BIT_ENC_DEPTH", 1), dec_depth=cfg.get("BIT_DEC_DEPTH", 8),
        heads=cfg.get("BIT_HEADS", 8), base=cfg.get("unet_base", 32)),
    "changeformer": lambda cfg: ChangeFormer(
        in_ch=cfg.get("IN_CHANNELS", IN_CHANNELS),
        embed_dims=tuple(cfg.get("CHANGEFORMER_EMBED_DIMS", [32, 64, 160, 256])),
        depths=tuple(cfg.get("CHANGEFORMER_DEPTHS", [2, 2, 2, 2])),
        heads=tuple(cfg.get("CHANGEFORMER_HEADS", [1, 2, 5, 8])),
        decoder_dim=cfg.get("CHANGEFORMER_DECODER_DIM", 256)),
    "changemamba": lambda cfg: ChangeMambaWrapper(
        variant=cfg.get("CHANGEMAMBA_VARIANT", "tiny"),
        encoder_pretrained=cfg.get("CHANGEMAMBA_ENCODER_PRETRAINED", None),
        in_chans=cfg.get("IN_CHANNELS", IN_CHANNELS),
    ),
}

def build_model(config: Dict[str, Any]) -> nn.Module:
    name = config.get("MODEL_NAME", MODEL_NAME)
    if name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown MODEL_NAME '{name}'. Available: {list(MODEL_REGISTRY.keys())}")
    model = MODEL_REGISTRY[name](config)
    return model

_test_model = build_model(CONFIG)
n_params = sum(p.numel() for p in _test_model.parameters())
print(f"Built '{MODEL_NAME}' with {n_params:,} parameters.")
del _test_model

Built 'siamese_unet' with 16,908,001 parameters.


In [20]:
# Cell 17 — Losses (all masked to valid pixels)

def masked_bce(logits, target, valid_mask, eps=1e-7):
    loss = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    loss = loss * valid_mask.float()
    denom = valid_mask.float().sum().clamp_min(1.0)
    return loss.sum() / denom

def masked_dice(logits, target, valid_mask, eps=1e-6):
    prob = torch.sigmoid(logits)
    prob = prob * valid_mask.float(); tgt = target * valid_mask.float()
    inter = (prob * tgt).sum(dim=(1, 2, 3))
    union = prob.sum(dim=(1, 2, 3)) + tgt.sum(dim=(1, 2, 3))
    dice = (2 * inter + eps) / (union + eps)
    return 1 - dice.mean()

def masked_focal(logits, target, valid_mask, gamma=2.0, alpha=0.25, eps=1e-7):
    prob = torch.sigmoid(logits)
    ce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    p_t = prob * target + (1 - prob) * (1 - target)
    alpha_t = alpha * target + (1 - alpha) * (1 - target)
    loss = alpha_t * (1 - p_t).clamp_min(eps).pow(gamma) * ce
    loss = loss * valid_mask.float()
    denom = valid_mask.float().sum().clamp_min(1.0)
    return loss.sum() / denom

def masked_tversky(logits, target, valid_mask, alpha=0.3, beta=0.7, eps=1e-6):
    prob = torch.sigmoid(logits)
    prob = prob * valid_mask.float(); tgt = target * valid_mask.float()
    tp = (prob * tgt).sum(dim=(1, 2, 3))
    fp = (prob * (1 - tgt)).sum(dim=(1, 2, 3))
    fn = ((1 - prob) * tgt).sum(dim=(1, 2, 3))
    tversky = (tp + eps) / (tp + alpha * fp + beta * fn + eps)
    return 1 - tversky.mean()

def build_loss(config: Dict[str, Any]):
    name = config.get("LOSS_NAME", LOSS_NAME)
    bce_w, dice_w = config.get("BCE_WEIGHT", 0.5), config.get("DICE_WEIGHT", 0.5)
    gamma, alpha = config.get("FOCAL_GAMMA", 2.0), config.get("FOCAL_ALPHA", 0.25)
    t_a, t_b = config.get("TVERSKY_ALPHA", 0.3), config.get("TVERSKY_BETA", 0.7)

    def loss_fn(logits, target, valid_mask):
        if name == "bce":
            return masked_bce(logits, target, valid_mask)
        if name == "dice":
            return masked_dice(logits, target, valid_mask)
        if name == "bce_dice":
            return bce_w * masked_bce(logits, target, valid_mask) + dice_w * masked_dice(logits, target, valid_mask)
        if name == "focal":
            return masked_focal(logits, target, valid_mask, gamma, alpha)
        if name == "focal_dice":
            return masked_focal(logits, target, valid_mask, gamma, alpha) + masked_dice(logits, target, valid_mask)
        if name == "tversky":
            return masked_tversky(logits, target, valid_mask, t_a, t_b)
        if name == "bce_tversky":
            return bce_w * masked_bce(logits, target, valid_mask) + dice_w * masked_tversky(logits, target, valid_mask, t_a, t_b)
        if name == "focal_tversky":
            return masked_focal(logits, target, valid_mask, gamma, alpha) + masked_tversky(logits, target, valid_mask, t_a, t_b)
        raise ValueError(f"Unknown LOSS_NAME '{name}'")
    return loss_fn

LOSS_FN = build_loss(CONFIG)

# Sanity check: invalid pixels must not affect the loss.
with torch.no_grad():
    logits = torch.randn(2, 1, 16, 16)
    target = torch.randint(0, 2, (2, 1, 16, 16)).float()
    vmask_all_valid = torch.ones(2, 1, 16, 16, dtype=torch.bool)
    vmask_some_invalid = vmask_all_valid.clone()
    vmask_some_invalid[:, :, :4, :4] = False
    target_perturbed = target.clone()
    target_perturbed[:, :, :4, :4] = 1 - target_perturbed[:, :, :4, :4]  # corrupt only the invalid region
    l1 = LOSS_FN(logits, target, vmask_some_invalid)
    l2 = LOSS_FN(logits, target_perturbed, vmask_some_invalid)
    assert torch.allclose(l1, l2, atol=1e-5), "FAIL: invalid pixels are leaking into the loss!"
    print(f"Sanity check passed: corrupting invalid-region targets changed loss by {abs((l1-l2).item()):.2e} (~0).")
print(f"Using loss: {LOSS_NAME}")


Sanity check passed: corrupting invalid-region targets changed loss by 0.00e+00 (~0).
Using loss: bce_dice


In [21]:
# Cell 18 — Metrics (valid-pixel only; accumulated confusion counts for dataset-level metrics)

@dataclass
class ConfusionAccumulator:
    tp: int = 0; fp: int = 0; fn: int = 0; tn: int = 0

    def update(self, pred_bin: np.ndarray, target: np.ndarray, valid_mask: np.ndarray):
        p = pred_bin[valid_mask].astype(bool); t = target[valid_mask].astype(bool)
        self.tp += int(np.sum(p & t)); self.fp += int(np.sum(p & ~t))
        self.fn += int(np.sum(~p & t)); self.tn += int(np.sum(~p & ~t))

    def compute(self, eps=1e-9) -> Dict[str, float]:
        tp, fp, fn, tn = self.tp, self.fp, self.fn, self.tn
        n = tp + fp + fn + tn
        iou = tp / (tp + fp + fn + eps)
        dice = 2 * tp / (2 * tp + fp + fn + eps)
        precision = tp / (tp + fp + eps)
        recall = tp / (tp + fn + eps)
        specificity = tn / (tn + fp + eps)
        accuracy = (tp + tn) / (n + eps)
        balanced_acc = 0.5 * (recall + specificity)
        return {"iou": iou, "dice_f1": dice, "precision": precision, "recall": recall,
                "specificity": specificity, "accuracy": accuracy, "balanced_accuracy": balanced_acc,
                "tp": tp, "fp": fp, "fn": fn, "tn": tn}


def per_image_metrics(prob: np.ndarray, target: np.ndarray, valid_mask: np.ndarray, threshold: float) -> Dict[str, float]:
    pred_bin = (prob >= threshold)
    acc = ConfusionAccumulator(); acc.update(pred_bin, target, valid_mask)
    m = acc.compute()
    y_true = target[valid_mask].astype(int); y_prob = prob[valid_mask]
    try:
        m["kappa"] = cohen_kappa_score(y_true, pred_bin[valid_mask].astype(int)) if len(np.unique(y_true)) > 1 else np.nan
        m["mcc"] = matthews_corrcoef(y_true, pred_bin[valid_mask].astype(int)) if len(np.unique(y_true)) > 1 else np.nan
        m["pr_auc"] = average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
        m["roc_auc"] = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    except Exception:
        m["kappa"] = m["mcc"] = m["pr_auc"] = m["roc_auc"] = np.nan
    return m

print("Metrics module ready: ConfusionAccumulator, per_image_metrics.")


Metrics module ready: ConfusionAccumulator, per_image_metrics.


In [22]:
# Cell 19 — Checkpoint save/load utilities

def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, best_val_score, seed, threshold, config):
    state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
    ckpt = {
        "model_state_dict": state,
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "amp_scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "epoch": epoch,
        "best_val_score": best_val_score,
        "model_name": config.get("MODEL_NAME", MODEL_NAME),
        "config": config,
        "in_channels": int(config.get("IN_CHANNELS", IN_CHANNELS)),
        "seed": seed,
        "threshold": float(threshold),
        "preprocessing": PREPROCESSOR.to_dict(),
    }
    torch.save(ckpt, path)

def load_checkpoint(path, device="cpu"):
    return torch.load(path, map_location=device)

def load_model_from_checkpoint(path, device):
    ckpt = load_checkpoint(path, device)
    cfg = ckpt.get("config", CONFIG)
    model = build_model(cfg).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model, ckpt

# Sanity test: save/load roundtrip
_m = build_model(CONFIG)
_opt = torch.optim.AdamW(_m.parameters(), lr=1e-4)
_test_path = os.path.join(DIRS["checkpoints"], "_sanity_test.pt")
save_checkpoint(_test_path, _m, _opt, None, None, epoch=0, best_val_score=0.0, seed=SEED, threshold=0.5, config=CONFIG)
_m2, _ckpt2 = load_model_from_checkpoint(_test_path, "cpu")
assert _ckpt2["preprocessing"] == PREPROCESSOR.to_dict()
assert _ckpt2["in_channels"] == IN_CHANNELS
assert all(torch.equal(a, b) for a, b in zip(_m.state_dict().values(), _m2.state_dict().values()))
os.remove(_test_path)
print("Checkpoint save/load sanity test passed.")

Checkpoint save/load sanity test passed.


In [23]:
# Cell 20 — Train / validation epoch functions with gradient accumulation

def run_epoch(model, loader, optimizer, scaler, loss_fn, device, train: bool, use_amp: bool, accum_steps: int = ACCUM_STEPS):
    model.train(mode=train)
    total_loss, n_batches = 0.0, 0
    acc = ConfusionAccumulator()
    accum_steps = max(1, int(accum_steps)) if train else 1
    ctx = torch.enable_grad() if train else torch.no_grad()

    if train:
        optimizer.zero_grad(set_to_none=True)

    with ctx:
        for batch_idx, batch in enumerate(loader):
            pre = batch["pre"].to(device, non_blocking=True)
            post = batch["post"].to(device, non_blocking=True)
            target = batch["target"].unsqueeze(1).to(device, non_blocking=True)
            valid_mask = batch["valid_mask"].unsqueeze(1).to(device, non_blocking=True)

            with autocast(device_type="cuda", enabled=use_amp and device.type == "cuda"):
                logits = model(pre, post)
                raw_loss = loss_fn(logits, target, valid_mask)
                loss_for_backward = raw_loss / accum_steps

            if train:
                if use_amp:
                    scaler.scale(loss_for_backward).backward()
                else:
                    loss_for_backward.backward()

                should_step = ((batch_idx + 1) % accum_steps == 0) or ((batch_idx + 1) == len(loader))
                if should_step:
                    if use_amp:
                        if GRAD_CLIP_NORM is not None:
                            scaler.unscale_(optimizer)
                            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                        scaler.step(optimizer); scaler.update()
                    else:
                        if GRAD_CLIP_NORM is not None:
                            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                        optimizer.step()
                    optimizer.zero_grad(set_to_none=True)

            with torch.no_grad():
                prob = torch.sigmoid(logits)
                pred_bin = (prob >= THRESHOLD).cpu().numpy().astype(bool)
                acc.update(pred_bin, target.cpu().numpy().astype(bool), valid_mask.cpu().numpy().astype(bool))

            total_loss += raw_loss.item(); n_batches += 1

    metrics = acc.compute()
    metrics["loss"] = total_loss / max(n_batches, 1)
    return metrics

print(f"run_epoch() ready with ACCUM_STEPS={ACCUM_STEPS}.")

run_epoch() ready with ACCUM_STEPS=4.


In [24]:
# Cell 21 — Multi-GPU runner (DDP with automatic DataParallel / single-GPU fallback)

def build_optimizer_scheduler(model, config, steps_per_epoch):
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["LEARNING_RATE"], weight_decay=config["WEIGHT_DECAY"])
    if config.get("SCHEDULER_NAME", "cosine") == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["EPOCHS"])
    else:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)
    return optimizer, scheduler

def ddp_worker(rank, world_size, seed, config, result_queue):
    os.environ.setdefault("MASTER_ADDR", "127.0.0.1")
    os.environ.setdefault("MASTER_PORT", "29500")
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)
    device = torch.device(f"cuda:{rank}")
    set_global_seed(seed)

    model = build_model(config).to(device)
    model = TorchDDP(model, device_ids=[rank])

    train_sampler = DistributedSampler(TRAIN_DS, num_replicas=world_size, rank=rank, shuffle=True, seed=seed)
    train_loader = make_loader(TRAIN_DS, config["BATCH_SIZE_PER_GPU"], shuffle=True, sampler=train_sampler)
    val_loader = VAL_LOADER  # validated on rank 0 only, below

    optimizer, scheduler = build_optimizer_scheduler(model, config, len(train_loader))
    scaler = GradScaler(
        "cuda",
        enabled=config["USE_AMP"] and device.type == "cuda"
    )
    loss_fn = build_loss(config)

    best_val_iou, patience_ctr, history = -1.0, 0, []
    for epoch in range(config["EPOCHS"]):
        train_sampler.set_epoch(epoch)
        t0 = time.time()
        train_m = run_epoch(model, train_loader, optimizer, scaler, loss_fn, device, train=True, use_amp=config["USE_AMP"], accum_steps=config.get("ACCUM_STEPS", ACCUM_STEPS))
        dist.barrier()

        if rank == 0:
            val_m = run_epoch(model, val_loader, None, None, loss_fn, device, train=False, use_amp=config["USE_AMP"], accum_steps=1)
            if config.get("SCHEDULER_NAME") == "plateau":
                scheduler.step(val_m["iou"])
            else:
                scheduler.step()
            epoch_time = time.time() - t0
            row = {"epoch": epoch, "train_loss": train_m["loss"], "val_loss": val_m["loss"],
                   "train_iou": train_m["iou"], "val_iou": val_m["iou"],
                   "train_dice": train_m["dice_f1"], "val_dice": val_m["dice_f1"],
                   "lr": optimizer.param_groups[0]["lr"], "epoch_time_sec": epoch_time}
            history.append(row)
            save_checkpoint(os.path.join(DIRS["checkpoints"], "latest_model.pt"), model, optimizer, scheduler,
                             scaler, epoch, best_val_iou, seed, THRESHOLD, config)
            if val_m["iou"] > best_val_iou:
                best_val_iou = val_m["iou"]; patience_ctr = 0
                save_checkpoint(os.path.join(DIRS["checkpoints"], "best_model.pt"), model, optimizer, scheduler,
                                 scaler, epoch, best_val_iou, seed, THRESHOLD, config)
            else:
                patience_ctr += 1
            print(f"[epoch {epoch:03d}] train_loss={train_m['loss']:.4f} val_loss={val_m['loss']:.4f} "
                  f"train_iou={train_m['iou']:.4f} val_iou={val_m['iou']:.4f} best={best_val_iou:.4f} "
                  f"patience={patience_ctr}/{config['EARLY_STOPPING_PATIENCE']} time={epoch_time:.1f}s")
            if patience_ctr >= config["EARLY_STOPPING_PATIENCE"]:
                print("Early stopping triggered."); stop_flag = torch.tensor(1, device=device)
            else:
                stop_flag = torch.tensor(0, device=device)
        else:
            stop_flag = torch.tensor(0, device=device)
        dist.broadcast(stop_flag, src=0)
        if stop_flag.item() == 1:
            break

    if rank == 0:
        pd.DataFrame(history).to_csv(os.path.join(DIRS["logs"], "training_history.csv"), index=False)
        result_queue.put(history)
    dist.destroy_process_group()


def train_single_process(config, device, seed):
    """Fallback path: single GPU, or nn.DataParallel across multiple GPUs on one process."""
    model = build_model(config).to(device)
    if EFFECTIVE_DP:
        model = nn.DataParallel(model)
        print(f"Using nn.DataParallel across {N_GPUS} GPUs.")
    effective_bs = config["BATCH_SIZE_PER_GPU"] * (N_GPUS if EFFECTIVE_DP else 1)
    train_loader = make_loader(TRAIN_DS, effective_bs, shuffle=True)

    optimizer, scheduler = build_optimizer_scheduler(model, config, len(train_loader))
    scaler = GradScaler(
        "cuda",
        enabled=config["USE_AMP"] and device.type == "cuda"
    )
    loss_fn = build_loss(config)

    start_epoch = 0
    best_val_iou = -1.0
    if config.get("RESUME_CHECKPOINT"):
        ckpt = load_checkpoint(config["RESUME_CHECKPOINT"], device)
        (model.module if hasattr(model, "module") else model).load_state_dict(ckpt["model_state_dict"])
        if ckpt.get("optimizer_state_dict"): optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if ckpt.get("scheduler_state_dict") and scheduler is not None:
            try: scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            except Exception: pass
        start_epoch = ckpt.get("epoch", 0) + 1
        best_val_iou = ckpt.get("best_val_score", -1.0)
        print(f"Resumed from {config['RESUME_CHECKPOINT']} at epoch {start_epoch}, best_val_iou={best_val_iou:.4f}")

    history, patience_ctr = [], 0
    for epoch in range(start_epoch, config["EPOCHS"]):
        t0 = time.time()
        train_m = run_epoch(model, train_loader, optimizer, scaler, loss_fn, device, train=True,
                             use_amp=config["USE_AMP"], accum_steps=config.get("ACCUM_STEPS", ACCUM_STEPS))
        val_m = run_epoch(model, VAL_LOADER, None, None, loss_fn, device, train=False, use_amp=config["USE_AMP"], accum_steps=1)
        if config.get("SCHEDULER_NAME") == "plateau":
            scheduler.step(val_m["iou"])
        else:
            scheduler.step()
        epoch_time = time.time() - t0
        row = {"epoch": epoch, "train_loss": train_m["loss"], "val_loss": val_m["loss"],
               "train_iou": train_m["iou"], "val_iou": val_m["iou"],
               "train_dice": train_m["dice_f1"], "val_dice": val_m["dice_f1"],
               "lr": optimizer.param_groups[0]["lr"], "epoch_time_sec": epoch_time}
        history.append(row)
        save_checkpoint(os.path.join(DIRS["checkpoints"], "latest_model.pt"), model, optimizer, scheduler,
                         scaler, epoch, best_val_iou, seed, THRESHOLD, config)
        if val_m["iou"] > best_val_iou:
            best_val_iou = val_m["iou"]; patience_ctr = 0
            save_checkpoint(os.path.join(DIRS["checkpoints"], "best_model.pt"), model, optimizer, scheduler,
                             scaler, epoch, best_val_iou, seed, THRESHOLD, config)
        else:
            patience_ctr += 1
        print(f"[epoch {epoch:03d}] train_loss={train_m['loss']:.4f} val_loss={val_m['loss']:.4f} "
              f"train_iou={train_m['iou']:.4f} val_iou={val_m['iou']:.4f} best={best_val_iou:.4f} "
              f"patience={patience_ctr}/{config['EARLY_STOPPING_PATIENCE']} time={epoch_time:.1f}s")
        if patience_ctr >= config["EARLY_STOPPING_PATIENCE"]:
            print("Early stopping triggered."); break

    pd.DataFrame(history).to_csv(os.path.join(DIRS["logs"], "training_history.csv"), index=False)
    return history

def build_optimizer_scheduler(model, config, steps_per_epoch):
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["LEARNING_RATE"], weight_decay=config["WEIGHT_DECAY"])
    warmup_epochs = config.get("WARMUP_EPOCHS", 0)
    if warmup_epochs > 0:
        warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["EPOCHS"] - warmup_epochs)
        scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs])
    elif config.get("SCHEDULER_NAME", "cosine") == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["EPOCHS"])
    else:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)
    return optimizer, scheduler


In [25]:
# # Cell 22 — Launch training for RUN_SEEDS[0] (development run). Multi-seed loop is in Cell 32.
# # NOTE: true torch.distributed.launch-style DDP does not run cleanly inside a single Jupyter/Kaggle cell via
# # mp.spawn in all Kaggle kernel configurations; we attempt it and cleanly fall back to DataParallel/single-GPU
# # if spawning fails, per the notebook's requirement to "provide a clean nn.DataParallel fallback".

# def train_one_seed(seed: int, config: Dict[str, Any]) -> List[Dict[str, Any]]:
#     set_global_seed(seed)
#     if EFFECTIVE_DDP:
#         try:
#             world_size = N_GPUS
#             ctx = mp.get_context("spawn")
#             result_queue = ctx.Queue()
#             mp.spawn(ddp_worker, args=(world_size, seed, config, result_queue), nprocs=world_size, join=True)
#             history = result_queue.get() if not result_queue.empty() else \
#                 pd.read_csv(os.path.join(DIRS["logs"], "training_history.csv")).to_dict("records")
#             return history
#         except Exception as e:
#             print(f"DDP launch failed ({e}); falling back to single-process "
#                   f"{'DataParallel' if N_GPUS > 1 else 'single-GPU'} training.")
#     return train_single_process(config, DEVICE, seed)

# HISTORY = train_one_seed(RUN_SEEDS[0], CONFIG)
# print(f"Training complete for seed {RUN_SEEDS[0]}. Epochs run: {len(HISTORY)}.")


In [26]:
# Cell 23 — Training curves

hist_df = pd.read_csv(os.path.join(DIRS["logs"], "training_history.csv"))

def plot_curve(df, cols, ylabel, title, fname):
    fig, ax = plt.subplots(figsize=(8, 5))
    for c in cols:
        ax.plot(df["epoch"], df[c], label=c)
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel); ax.set_title(title); ax.legend()
    for ext in ["png", "pdf"]:
        fig.savefig(os.path.join(DIRS["figures"], f"{fname}.{ext}"), dpi=200, bbox_inches="tight")
    plt.close(fig)

plot_curve(hist_df, ["train_loss", "val_loss"], "Loss", "Training / Validation Loss", "loss_curve")
plot_curve(hist_df, ["train_iou", "val_iou"], "IoU", "Training / Validation IoU", "iou_curve")
plot_curve(hist_df, ["train_dice", "val_dice"], "Dice / F1", "Training / Validation Dice", "dice_curve")
plot_curve(hist_df, ["lr"], "Learning Rate", "Learning Rate Schedule", "lr_curve")
print("Saved loss/IoU/Dice/LR curves (PNG+PDF) to", DIRS["figures"])
hist_df.tail()


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/changeMamba_s1gfloods/logs/training_history.csv'

In [27]:
# Cell 24 — Load best checkpoint for evaluation

BEST_CKPT_PATH = os.path.join(DIRS["checkpoints"], "best_model.pt")
BEST_MODEL, BEST_CKPT = load_model_from_checkpoint(BEST_CKPT_PATH, DEVICE)
BEST_MODEL.to(DEVICE).eval()
print(f"Loaded best_model.pt (epoch {BEST_CKPT['epoch']}, best_val_score={BEST_CKPT['best_val_score']:.4f}, "
      f"seed={BEST_CKPT['seed']}, model={BEST_CKPT['model_name']}, in_channels={BEST_CKPT.get('in_channels')})")

@torch.no_grad()
def predict_prob_tensor(model, pre, post, use_tta=False, use_amp=USE_AMP):
    """Return [B,1,H,W] probabilities. TTA averages 4 rotations × optional horizontal flip (8 transforms)."""
    if not use_tta:
        with autocast(device_type="cuda", enabled=use_amp and pre.device.type == "cuda"):
            return torch.sigmoid(model(pre, post))

    probs = []
    for k in range(4):
        for hflip in (False, True):
            pre_t = torch.rot90(pre, k, dims=(-2, -1))
            post_t = torch.rot90(post, k, dims=(-2, -1))
            if hflip:
                pre_t = torch.flip(pre_t, dims=(-1,)); post_t = torch.flip(post_t, dims=(-1,))
            with autocast(device_type="cuda", enabled=use_amp and pre.device.type == "cuda"):
                p = torch.sigmoid(model(pre_t, post_t))
            if hflip:
                p = torch.flip(p, dims=(-1,))
            p = torch.rot90(p, -k, dims=(-2, -1))
            probs.append(p)
    return torch.stack(probs, dim=0).mean(dim=0)

@torch.no_grad()
def collect_probs_targets(model, loader, device):
    all_probs, all_targets, all_valid, all_keys = [], [], [], []
    for batch in tqdm(loader, desc=f"Collecting predictions (TTA={INFERENCE_TTA})"):
        pre = batch["pre"].to(device); post = batch["post"].to(device)
        prob = predict_prob_tensor(model, pre, post, use_tta=INFERENCE_TTA).squeeze(1).cpu().numpy()
        target = batch["target"].numpy(); valid = batch["valid_mask"].numpy()
        for i in range(prob.shape[0]):
            all_probs.append(prob[i]); all_targets.append(target[i]); all_valid.append(valid[i]); all_keys.append(batch["key"][i])
    return all_probs, all_targets, all_valid, all_keys

Loaded best_model.pt (epoch 150, best_val_score=0.9038, seed=42, model=siamese_unet, in_channels=1)


In [28]:
# Cell 25 — Validation-only threshold search (never uses test/external data)

val_probs, val_targets, val_valid, val_keys = collect_probs_targets(BEST_MODEL, VAL_LOADER, DEVICE)
thresholds = np.linspace(*THRESHOLD_SEARCH_RANGE, THRESHOLD_SEARCH_STEPS)
search_rows = []
for th in thresholds:
    acc = ConfusionAccumulator()
    for p, t, v in zip(val_probs, val_targets, val_valid):
        acc.update(p >= th, t.astype(bool), v.astype(bool))
    search_rows.append({"threshold": th, **acc.compute()})

threshold_search_df = pd.DataFrame(search_rows)
threshold_search_df.to_csv(os.path.join(DIRS["metrics"], "validation_threshold_search.csv"), index=False)
best_row = threshold_search_df.loc[threshold_search_df["iou"].idxmax()]
BEST_THRESHOLD = float(best_row["threshold"])
threshold_record = {"best_threshold": BEST_THRESHOLD, "criterion": "val_iou", "val_iou_at_best": float(best_row["iou"]),
                    "tta_used_for_validation": bool(INFERENCE_TTA)}
with open(os.path.join(DIRS["metrics"], "best_threshold.json"), "w") as f:
    json.dump(threshold_record, f, indent=2)
with open(os.path.join(DIRS["checkpoints"], "best_threshold.json"), "w") as f:
    json.dump(threshold_record, f, indent=2)

# FIX: persist the validation-selected threshold into the checkpoint actually used by predict_geotiff().
_ckpt_locked = load_checkpoint(BEST_CKPT_PATH, "cpu")
_ckpt_locked["threshold"] = BEST_THRESHOLD
_ckpt_locked["threshold_selection"] = threshold_record
torch.save(_ckpt_locked, BEST_CKPT_PATH)
BEST_CKPT = _ckpt_locked

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(threshold_search_df["threshold"], threshold_search_df["iou"], label="IoU")
ax.plot(threshold_search_df["threshold"], threshold_search_df["dice_f1"], label="Dice")
ax.axvline(BEST_THRESHOLD, linestyle="--", label=f"best={BEST_THRESHOLD:.3f}")
ax.set_xlabel("Threshold"); ax.set_ylabel("Score"); ax.legend(); ax.set_title("Validation Threshold Search")
fig.savefig(os.path.join(DIRS["figures"], "threshold_curve.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

THRESHOLD = BEST_THRESHOLD
print(f"Locked THRESHOLD = {THRESHOLD:.3f} (validation IoU={best_row['iou']:.4f}) and persisted it into best_model.pt.")

Locked THRESHOLD = 0.900 (validation IoU=0.9155) and persisted it into best_model.pt.


In [29]:
# Cell 26 — Q1-level test-set evaluation (locked threshold, best checkpoint, TEST split only)

test_probs, test_targets, test_valid, test_keys = collect_probs_targets(BEST_MODEL, TEST_LOADER, DEVICE)

# Dataset-level (globally accumulated confusion counts)
global_acc = ConfusionAccumulator()
for p, t, v in zip(test_probs, test_targets, test_valid):
    global_acc.update(p >= THRESHOLD, t.astype(bool), v.astype(bool))
global_metrics = global_acc.compute()

all_true_flat = np.concatenate([t[v].astype(int) for t, v in zip(test_targets, test_valid)])
all_prob_flat = np.concatenate([p[v] for p, v in zip(test_probs, test_valid)])
all_pred_flat = (all_prob_flat >= THRESHOLD).astype(int)

global_metrics["kappa"] = cohen_kappa_score(all_true_flat, all_pred_flat)
global_metrics["mcc"] = matthews_corrcoef(all_true_flat, all_pred_flat)
global_metrics["pr_auc"] = average_precision_score(all_true_flat, all_prob_flat)
global_metrics["roc_auc"] = roc_auc_score(all_true_flat, all_prob_flat)
global_metrics["balanced_accuracy_sklearn"] = balanced_accuracy_score(all_true_flat, all_pred_flat)
global_metrics["threshold"] = THRESHOLD
global_metrics["n_test_images"] = len(test_keys)

test_metrics_summary = pd.DataFrame([global_metrics])
test_metrics_summary.to_csv(os.path.join(DIRS["metrics"], "test_metrics_summary.csv"), index=False)

# Per-image metrics
per_image_rows = []
for k, p, t, v in zip(test_keys, test_probs, test_targets, test_valid):
    m = per_image_metrics(p, t, v, THRESHOLD); m["key"] = k
    per_image_rows.append(m)
per_image_df = pd.DataFrame(per_image_rows)
per_image_df.to_csv(os.path.join(DIRS["metrics"], "test_metrics_per_image.csv"), index=False)

# Confusion matrix (dataset level)
cm = np.array([[global_metrics["tn"], global_metrics["fp"]], [global_metrics["fn"], global_metrics["tp"]]])
pd.DataFrame(cm, index=["true_0", "true_1"], columns=["pred_0", "pred_1"]).to_csv(
    os.path.join(DIRS["metrics"], "test_confusion_matrix.csv"))

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["pred_non-water", "pred_water"], yticklabels=["true_non-water", "true_water"])
ax.set_title("Test Confusion Matrix (valid pixels, dataset-level)")
fig.savefig(os.path.join(DIRS["metrics"], "test_confusion_matrix.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

print("TEST SET RESULTS (dataset-level, valid pixels only):")
for k in ["iou", "dice_f1", "precision", "recall", "specificity", "accuracy", "balanced_accuracy",
          "kappa", "mcc", "pr_auc", "roc_auc"]:
    print(f"  {k:>20s}: {global_metrics[k]:.4f}")

print("CAVEAT: S1GFloods filenames carry no event ID, so event-level split independence cannot be verified; "
      "benchmark test scores may be optimistic relative to external Gaibandha performance.")


TEST SET RESULTS (dataset-level, valid pixels only):
                   iou: 0.8142
               dice_f1: 0.8976
             precision: 0.8784
                recall: 0.9176
           specificity: 0.9693
              accuracy: 0.9592
     balanced_accuracy: 0.9434
                 kappa: 0.8721
                   mcc: 0.8724
                pr_auc: 0.9522
               roc_auc: 0.9836
CAVEAT: S1GFloods filenames carry no event ID, so event-level split independence cannot be verified; benchmark test scores may be optimistic relative to external Gaibandha performance.


In [30]:
# Cell 27 — OPTIONAL external reference-mask evaluation (run/re-run after Cell 35 external inference)
# Strictly compares on the existing raster grid; never resizes/reprojects the prediction or reference.

_expected_external_mask = globals().get("mask_path") or os.path.join(DIRS["inference_outputs"], "gaibandha", "flood_binary_mask.tif")
if EXTERNAL_REF_MASK is None:
    print("EXTERNAL_REF_MASK=None — optional external evaluation skipped.")
elif not os.path.exists(_expected_external_mask):
    print("External prediction mask not found yet. Run Cell 35, then re-run this cell:", _expected_external_mask)
elif not os.path.exists(EXTERNAL_REF_MASK):
    print("Configured EXTERNAL_REF_MASK does not exist:", EXTERNAL_REF_MASK)
else:
    with rasterio.open(_expected_external_mask) as pred_src, rasterio.open(EXTERNAL_REF_MASK) as ref_src:
        same_grid = (pred_src.width == ref_src.width and pred_src.height == ref_src.height and
                     pred_src.crs == ref_src.crs and pred_src.transform.almost_equals(ref_src.transform, precision=1e-9))
        if not same_grid:
            raise RuntimeError("External reference mask grid does not exactly match the prediction. No silent resampling is allowed.")
        pred = pred_src.read(1)
        ref = ref_src.read(1).astype(np.float32)
        valid = (pred != 255) & np.isfinite(ref)
        if ref_src.nodata is not None:
            valid &= ref != ref_src.nodata
        ref_pos = (ref > 0) if EXTERNAL_REF_POSITIVE_VALUE is None else (ref == EXTERNAL_REF_POSITIVE_VALUE)
        pred_pos = pred == 1
        acc = ConfusionAccumulator()
        acc.update(pred_pos, ref_pos, valid)
        ext_ref_metrics = acc.compute()
        pd.DataFrame([{k: ext_ref_metrics[k] for k in ["iou", "dice_f1", "precision", "recall", "tp", "fp", "fn", "tn"]}]).to_csv(
            os.path.join(DIRS["metrics"], "external_reference_metrics.csv"), index=False)
        print("External reference evaluation:")
        for k in ["iou", "dice_f1", "precision", "recall"]:
            print(f"  {k:>10s}: {ext_ref_metrics[k]:.4f}")


EXTERNAL_REF_MASK=None — optional external evaluation skipped.


In [31]:
# Cell 28 — Image-level bootstrap confidence intervals (test set)

def bootstrap_ci(per_image_df: pd.DataFrame, metrics: List[str], n_reps: int, ci: float, seed: int = 0):
    rng = np.random.RandomState(seed)
    n = len(per_image_df)
    rows = []
    boot_samples = {m: [] for m in metrics}
    for _ in range(n_reps):
        idx = rng.randint(0, n, size=n)  # bootstrap at the IMAGE level, not pixel level
        sample = per_image_df.iloc[idx]
        for m in metrics:
            boot_samples[m].append(np.nanmean(sample[m].values))
    alpha = (1 - ci) / 2
    for m in metrics:
        vals = np.array(boot_samples[m])
        rows.append({
            "metric": m,
            "point_estimate_mean_of_images": float(np.nanmean(per_image_df[m])),
            "bootstrap_mean": float(np.nanmean(vals)),
            f"ci_lower_{int(ci*100)}": float(np.nanpercentile(vals, alpha * 100)),
            f"ci_upper_{int(ci*100)}": float(np.nanpercentile(vals, (1 - alpha) * 100)),
            "n_bootstrap_reps": n_reps,
        })
    return pd.DataFrame(rows)

bootstrap_df = bootstrap_ci(per_image_df, ["iou", "dice_f1", "precision", "recall"],
                             n_reps=BOOTSTRAP_REPS, ci=BOOTSTRAP_CI, seed=SEED)
bootstrap_df.to_csv(os.path.join(DIRS["metrics"], "bootstrap_confidence_intervals.csv"), index=False)
bootstrap_df


,metric,point_estimate_mean_of_images,bootstrap_mean,ci_lower_95,ci_upper_95,n_bootstrap_reps
0,iou,0.810915,0.811332,0.796279,0.826687,1000
1,dice_f1,0.878126,0.878571,0.864694,0.892206,1000
2,precision,0.840071,0.840393,0.826216,0.855537,1000
3,recall,0.932849,0.933376,0.918350,0.946997,1000


In [32]:
# Cell 29 — Per-image analysis (distribution of scores, worst/best cases)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(per_image_df["iou"].dropna(), bins=20, ax=axes[0]); axes[0].set_title("Per-image IoU distribution")
sns.histplot(per_image_df["dice_f1"].dropna(), bins=20, ax=axes[1]); axes[1].set_title("Per-image Dice distribution")
fig.savefig(os.path.join(DIRS["figures"], "per_image_score_distributions.png"), dpi=200, bbox_inches="tight")
plt.close(fig)

worst5 = per_image_df.nsmallest(5, "iou")[["key", "iou", "dice_f1", "precision", "recall"]]
best5 = per_image_df.nlargest(5, "iou")[["key", "iou", "dice_f1", "precision", "recall"]]
print("Best 5 test images by IoU:\n", best5.to_string(index=False))
print("\nWorst 5 test images by IoU:\n", worst5.to_string(index=False))

worst5.to_csv(os.path.join(DIRS["metrics"], "worst5_test_images.csv"), index=False)
best5.to_csv(os.path.join(DIRS["metrics"], "best5_test_images.csv"), index=False)

Best 5 test images by IoU:
                 key      iou  dice_f1  precision   recall
test/image_4887.png 0.995357 0.997673   0.996640 0.998707
test/image_5131.png 0.987996 0.993962   0.993538 0.994385
test/image_4903.png 0.983836 0.991852   0.994157 0.989558
test/image_5265.png 0.983121 0.991489   0.992692 0.990289
test/image_4943.png 0.981349 0.990587   0.982930 0.998364

Worst 5 test images by IoU:
                 key  iou  dice_f1  precision  recall
test/image_4844.png  0.0      0.0        0.0     0.0
test/image_4925.png  0.0      0.0        0.0     0.0
test/image_4946.png  0.0      0.0        0.0     0.0
test/image_5158.png  0.0      0.0        0.0     0.0
test/image_5190.png  0.0      0.0        0.0     0.0


In [33]:
# Cell 30 — Publication-quality qualitative figures (random / best / worst)

def qualitative_panel(key, prob, target, valid, threshold, out_path):
    t = triplet_by_key[key]
    pre_gray = read_png_gray(t["pre"]); post_gray = read_png_gray(t["post"])
    diff_gray = post_gray.astype(np.float32) - pre_gray.astype(np.float32)

    pred_bin = prob >= threshold
    tp = pred_bin & (target == 1) & valid; fp = pred_bin & (target == 0) & valid
    fn = (~pred_bin) & (target == 1) & valid
    error_map = np.zeros((*target.shape, 3), dtype=np.float32)
    error_map[tp] = [0, 1, 0]; error_map[fp] = [1, 0, 0]; error_map[fn] = [0, 0, 1]

    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    panels = [
        (pre_gray, "PRE VV gray"), (post_gray, "POST VV gray"), (diff_gray, "POST-PRE gray"),
        (np.ma.masked_where(~valid, target), "Ground truth: new inundation"),
        (np.ma.masked_where(~valid, prob), "Probability map"),
        (np.ma.masked_where(~valid, pred_bin.astype(float)), "Binary prediction"),
        (error_map, "TP/FP/FN error map"), (valid.astype(float), "Valid mask"),
    ]
    for ax, (img, title) in zip(axes.flat, panels):
        if title == "POST-PRE gray":
            ax.imshow(img, cmap="RdBu", vmin=-80, vmax=80)
        elif title in {"PRE VV gray", "POST VV gray", "Valid mask"}:
            ax.imshow(img, cmap="gray")
        elif title == "Probability map":
            ax.imshow(img, cmap="viridis", vmin=0, vmax=1)
        elif title == "TP/FP/FN error map":
            ax.imshow(img)
        else:
            ax.imshow(img, cmap="Blues", vmin=0, vmax=1)
        ax.set_title(title, fontsize=11); ax.axis("off")
    fig.suptitle(f"Qualitative example: {key}", fontsize=13)
    fig.tight_layout(); fig.savefig(out_path, dpi=250, bbox_inches="tight"); plt.close(fig)

test_iou_by_key = dict(zip(per_image_df["key"], per_image_df["iou"]))
sorted_keys = sorted(test_iou_by_key, key=lambda k: test_iou_by_key[k])
selection = {"worst": sorted_keys[:3], "best": sorted_keys[-3:], "random": random.sample(test_keys, min(3, len(test_keys)))}
key_to_arrays = {k: (p, t, v) for k, p, t, v in zip(test_keys, test_probs, test_targets, test_valid)}
for group, keys in selection.items():
    for k in keys:
        p, t, v = key_to_arrays[k]
        qualitative_panel(k, p, t, v, THRESHOLD, os.path.join(DIRS["qualitative"], f"{group}_{k.replace('/', '_')}.png"))
print(f"Saved qualitative figures to {DIRS['qualitative']} for groups: {list(selection.keys())}")

Saved qualitative figures to /kaggle/working/changeMamba_s1gfloods/qualitative for groups: ['worst', 'best', 'random']


In [34]:
# Cell 30A — Save qualitative prediction figures as EPS for Adobe Illustrator

EPS_DIR = os.path.join(DIRS["qualitative"], "eps")
os.makedirs(EPS_DIR, exist_ok=True)

for group, keys in selection.items():
    for k in keys:
        p, t, v = key_to_arrays[k]

        eps_path = os.path.join(
            EPS_DIR,
            f"{group}_{k.replace('/', '_')}.eps"
        )

        qualitative_panel(
            k,
            p,
            t,
            v,
            THRESHOLD,
            eps_path
        )

print(f"EPS prediction figures saved to: {EPS_DIR}")
print(f"Total EPS files: {len(glob.glob(os.path.join(EPS_DIR, '*.eps')))}")


EPS prediction figures saved to: /kaggle/working/changeMamba_s1gfloods/qualitative/eps
Total EPS files: 9


In [35]:
# Cell 31 — Model complexity and runtime report

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def model_size_mb(model):
    total_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    return total_bytes / (1024 ** 2)

total_params, trainable_params = count_params(BEST_MODEL)
size_mb = model_size_mb(BEST_MODEL)

try:
    from thop import profile
    dummy_pre = torch.randn(1, IN_CHANNELS, 256, 256).to(DEVICE)
    dummy_post = torch.randn(1, IN_CHANNELS, 256, 256).to(DEVICE)
    macs, _ = profile(BEST_MODEL, inputs=(dummy_pre, dummy_post), verbose=False)
    flops = macs * 2
except Exception as e:
    print(f"FLOPs/MACs profiling unavailable ({e}); leaving as NaN.")
    macs, flops = np.nan, np.nan

# Inference timing / throughput on a batch of validation tiles
BEST_MODEL.eval()
n_timing_batches = min(10, len(VAL_LOADER))
times = []
with torch.no_grad():
    it = iter(VAL_LOADER)
    for _ in range(n_timing_batches):
        batch = next(it)
        pre = batch["pre"].to(DEVICE); post = batch["post"].to(DEVICE)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t0 = time.time()
        _ = BEST_MODEL(pre, post)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        times.append((time.time() - t0) / pre.shape[0])
mean_infer_time_per_image = float(np.mean(times)) if times else np.nan
peak_mem_mb = torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 2) if DEVICE.type == "cuda" else np.nan

total_train_time = hist_df["epoch_time_sec"].sum() if "epoch_time_sec" in hist_df.columns else np.nan

complexity_df = pd.DataFrame([{
    "model_name": MODEL_NAME, "total_params": total_params, "trainable_params": trainable_params,
    "model_size_mb": size_mb, "macs": macs, "flops": flops,
    "mean_inference_time_sec_per_image": mean_infer_time_per_image,
    "throughput_images_per_sec": (1.0 / mean_infer_time_per_image) if mean_infer_time_per_image else np.nan,
    "peak_gpu_memory_mb": peak_mem_mb, "total_training_time_sec": total_train_time,
}])
complexity_df.to_csv(os.path.join(DIRS["metrics"], "complexity_runtime.csv"), index=False)
complexity_df


,model_name,total_params,trainable_params,model_size_mb,macs,flops,mean_inference_time_sec_per_image,throughput_images_per_sec,peak_gpu_memory_mb,total_training_time_sec
0,siamese_unet,16908001,16908001,64.498905,4.620655e+10,9.241310e+10,0.151379,6.60592,8983.234863,NaN


In [36]:
# Cell 32 — Multi-seed training + aggregation (set RUN_SEEDS = [42,52,62,72,82] in Cell 4 for the final run)
# For the default development config (RUN_SEEDS=[42]) this reuses the run already completed above and simply
# reports it as a single-seed "multi-seed" table; set RUN_SEEDS to multiple values and RE-RUN Cells 22-31 in a
# loop (or re-run this notebook end-to-end per seed) to populate genuinely independent runs before aggregating.

def evaluate_checkpoint_on_test(ckpt_path, threshold):
    model, ckpt = load_model_from_checkpoint(ckpt_path, DEVICE)
    model.eval()
    probs, targets, valid, keys = collect_probs_targets(model, TEST_LOADER, DEVICE)
    acc = ConfusionAccumulator()
    for p, t, v in zip(probs, targets, valid):
        acc.update(p >= threshold, t.astype(bool), v.astype(bool))
    m = acc.compute()
    return m, ckpt.get("seed")

multi_seed_rows = []
for seed in RUN_SEEDS:
    # In the single-seed default case this is the run we already have; for RUN_SEEDS with >1 entries, re-run
    # train_one_seed(seed, CONFIG) for each seed BEFORE this cell, saving per-seed checkpoints to distinct
    # subfolders (e.g. checkpoints/seed_{seed}/) to avoid overwriting best_model.pt between seeds.
    m, ckpt_seed = evaluate_checkpoint_on_test(BEST_CKPT_PATH, THRESHOLD)
    m["seed"] = seed
    multi_seed_rows.append(m)

multi_seed_df = pd.DataFrame(multi_seed_rows)
summary = {"metric": [], "mean": [], "std": []}
for col in ["iou", "dice_f1", "precision", "recall", "accuracy"]:
    summary["metric"].append(col)
    summary["mean"].append(multi_seed_df[col].mean())
    summary["std"].append(multi_seed_df[col].std())
multi_seed_summary_df = pd.DataFrame(summary)
multi_seed_df.to_csv(os.path.join(DIRS["metrics"], "multi_seed_raw_results.csv"), index=False)
multi_seed_summary_df.to_csv(os.path.join(DIRS["metrics"], "multi_seed_summary.csv"), index=False)
print(f"NOTE: with RUN_SEEDS={RUN_SEEDS}, {'this reflects a single training run' if len(RUN_SEEDS)==1 else 'each row should come from an independently trained checkpoint'}.")
multi_seed_summary_df


NOTE: with RUN_SEEDS=[42], this reflects a single training run.


,metric,mean,std
0,iou,0.814199,NaN
1,dice_f1,0.897585,NaN
2,precision,0.878439,NaN
3,recall,0.917584,NaN
4,accuracy,0.959196,NaN


In [37]:
# Cell 33 — Journal-ready results table

paper_row = {
    "Model": MODEL_NAME,
    "Input": f"Bi-temporal S1GFloods VV PRE+POST ({INPUT_MODE}, {IN_CHANNELS} ch/date)",
    "Loss": LOSS_NAME,
    "Parameters": total_params,
    "IoU": global_metrics["iou"],
    "Dice/F1": global_metrics["dice_f1"],
    "Precision": global_metrics["precision"],
    "Recall": global_metrics["recall"],
    "Specificity": global_metrics["specificity"],
    "Accuracy": global_metrics["accuracy"],
    "Balanced Accuracy": global_metrics["balanced_accuracy"],
    "Kappa": global_metrics["kappa"],
    "MCC": global_metrics["mcc"],
    "PR-AUC": global_metrics["pr_auc"],
    "Inference Time (s/image)": mean_infer_time_per_image,
}
if len(RUN_SEEDS) > 1:
    for m in ["iou", "dice_f1", "precision", "recall"]:
        row = multi_seed_summary_df[multi_seed_summary_df["metric"] == m].iloc[0]
        paper_row[f"{m}_mean_std"] = f"{row['mean']:.4f} ± {row['std']:.4f}"

paper_results_table = pd.DataFrame([paper_row])
paper_results_table.to_csv(os.path.join(DIRS["metrics"], "paper_results_table.csv"), index=False)
try:
    paper_results_table.to_excel(os.path.join(DIRS["metrics"], "paper_results_table.xlsx"), index=False)
except Exception as e:
    print(f"XLSX export skipped ({e}); CSV is available.")
paper_results_table


,Model,Input,Loss,Parameters,IoU,Dice/F1,Precision,Recall,Specificity,Accuracy,Balanced Accuracy,Kappa,MCC,PR-AUC,Inference Time (s/image)
0,siamese_unet,"Bi-temporal S1GFloods VV PRE+POST (1ch, 1 ch/d...",bce_dice,16908001,0.814199,0.897585,0.878439,0.917584,0.969268,0.959196,0.943426,0.872123,0.872446,0.952169,0.151379


In [38]:
# Cell 34 — Real Sentinel-1 GeoTIFF inference: VV name lookup, checkpoint preprocessing, safe tiling

def hann_window_2d(size: int) -> np.ndarray:
    w1 = np.hanning(size).astype(np.float32)
    w1[w1 == 0] = 1e-3
    return np.outer(w1, w1).astype(np.float32)


def get_vv_band_index(src, path_label="GeoTIFF") -> int:
    descriptions = list(src.descriptions or [])
    if not descriptions or all(d is None or str(d).strip() == "" for d in descriptions):
        print(f"WARNING: {path_label} has no band descriptions; UNVERIFIED fallback assumes band 1 is VV.")
        return 1
    names = [str(d).strip().upper() if d is not None else "" for d in descriptions]
    if "VV" not in names:
        raise RuntimeError(f"{path_label} has band descriptions {descriptions} but no band named 'VV'. Refusing to guess.")
    return names.index("VV") + 1


def _bounds_close(a, b, atol=1e-6):
    return np.allclose([a.left, a.bottom, a.right, a.top], [b.left, b.bottom, b.right, b.top], rtol=0, atol=atol)


def check_pre_post_compatibility(pre_path, post_path):
    with rasterio.open(pre_path) as p, rasterio.open(post_path) as q:
        mismatches = {}
        if p.width != q.width: mismatches["width"] = (p.width, q.width)
        if p.height != q.height: mismatches["height"] = (p.height, q.height)
        if p.crs != q.crs: mismatches["crs"] = (str(p.crs), str(q.crs))
        if not p.transform.almost_equals(q.transform, precision=1e-9): mismatches["transform"] = (p.transform, q.transform)
        if not np.allclose(p.res, q.res, rtol=0, atol=1e-9): mismatches["res"] = (p.res, q.res)
        if not _bounds_close(p.bounds, q.bounds): mismatches["bounds"] = (p.bounds, q.bounds)
        if mismatches:
            msg = "PRE/POST GeoTIFF compatibility check FAILED:\n" + "\n".join(f"  {k}: {v}" for k, v in mismatches.items())
            if AUTO_ALIGN:
                raise RuntimeError(msg + "\nAUTO_ALIGN=True does not trigger silent resampling; align explicitly outside this function.")
            raise RuntimeError(msg)

        pre_vv = get_vv_band_index(p, "PRE GeoTIFF")
        post_vv = get_vv_band_index(q, "POST GeoTIFF")
        if p.crs is not None and p.crs.is_geographic:
            print("WARNING: CRS is geographic (degrees). Pixel resolution is angular, not 10 m; area uses latitude-dependent row areas.")
        elif not (abs(abs(p.res[0]) - 10.0) < 0.5 and abs(abs(p.res[1]) - 10.0) < 0.5):
            print(f"WARNING: projected pixel resolution {p.res} differs from expected ~10 m.")
        print("PRE/POST compatibility PASSED: width/height/CRS/transform/resolution/bounds match; VV is available in both.")
        return {
            "width": p.width, "height": p.height, "crs": p.crs, "transform": p.transform, "res": p.res,
            "pre_vv_index": pre_vv, "post_vv_index": post_vv,
        }


def _external_calibration_db(vv_db: np.ndarray, preproc: S1GFloodsPreprocessor, mode: str) -> np.ndarray:
    """Optional identical PRE/POST two-point calibration. OFF by default; when on, output is approximate."""
    if mode == "none":
        return vv_db.astype(np.float32, copy=False)
    if mode != "two_point":
        raise ValueError("EXTERNAL_CALIBRATION must be 'none' or 'two_point'.")
    if CALIBRATION_STABLE_LAND_DB is None or CALIBRATION_RIVER_DB is None:
        raise ValueError("two_point calibration requires CALIBRATION_STABLE_LAND_DB and CALIBRATION_RIVER_DB.")
    x_water = float(CALIBRATION_RIVER_DB); x_land = float(CALIBRATION_STABLE_LAND_DB)
    if x_land == x_water:
        raise ValueError("Calibration stable-land and river dB anchors must differ.")
    y_water = preproc.lo + (CALIBRATION_TARGET_WATER_GRAY / 255.0) * (preproc.hi - preproc.lo)
    y_land = preproc.lo + (CALIBRATION_TARGET_LAND_GRAY / 255.0) * (preproc.hi - preproc.lo)
    a = (y_land - y_water) / (x_land - x_water)
    b = y_water - a * x_water
    out = a * vv_db.astype(np.float32) + b
    out[~np.isfinite(vv_db)] = np.nan
    return out.astype(np.float32)


def _safe_pad_2d(arr: np.ndarray, tile_size: int, constant_value: float = 0.0) -> np.ndarray:
    """Pad bottom/right to tile_size. Reflect only when requested pad is smaller than that dimension."""
    out = np.asarray(arr)
    pad_h = tile_size - out.shape[0]
    pad_w = tile_size - out.shape[1]
    if pad_h < 0 or pad_w < 0:
        raise ValueError(f"Array {out.shape} exceeds tile_size={tile_size}")
    if pad_h:
        mode = "reflect" if out.shape[0] > 1 and pad_h < out.shape[0] else "constant"
        kwargs = {} if mode == "reflect" else {"constant_values": constant_value}
        out = np.pad(out, ((0, pad_h), (0, 0)), mode=mode, **kwargs)
    if pad_w:
        mode = "reflect" if out.shape[1] > 1 and pad_w < out.shape[1] else "constant"
        kwargs = {} if mode == "reflect" else {"constant_values": constant_value}
        out = np.pad(out, ((0, 0), (0, pad_w)), mode=mode, **kwargs)
    return out


def _window_grid(width, height, block=2048):
    for y in range(0, height, block):
        for x in range(0, width, block):
            yield Window(x, y, min(block, width-x), min(block, height-y))


def gray_histogram_from_raster(src, vv_idx: int, preproc: S1GFloodsPreprocessor,
                               calibration_mode="none", scope_window: Optional[Window] = None) -> np.ndarray:
    """Streaming 256-bin histogram in the mapped gray domain; invalid pixels are excluded."""
    hist = np.zeros(256, dtype=np.int64)
    if scope_window is None:
        windows = _window_grid(src.width, src.height)
    else:
        x0, y0, w0, h0 = map(int, [scope_window.col_off, scope_window.row_off, scope_window.width, scope_window.height])
        windows = (Window(x0+x, y0+y, min(2048, w0-x), min(2048, h0-y))
                   for y in range(0, h0, 2048) for x in range(0, w0, 2048))
    for win in windows:
        raw = src.read(vv_idx, window=win).astype(np.float32)
        raw = _external_calibration_db(raw, preproc, calibration_mode)
        x, valid = preproc.from_db(raw, fill_gray=0.0)
        if valid.any():
            g = np.clip(np.rint(x[valid] * 255.0), 0, 255).astype(np.uint8)
            hist += np.bincount(g, minlength=256)
    return hist


def histogram_median_gray(hist: np.ndarray) -> float:
    total = int(hist.sum())
    if total == 0:
        return 0.0
    return float(np.searchsorted(np.cumsum(hist), (total + 1) / 2))


def _resolve_nodata_fill(src, vv_idx, preproc, nodata_fill, calibration_mode="none", scope_window=None):
    if nodata_fill == "zero":
        return 0.0
    if nodata_fill != "median":
        raise ValueError("NODATA_FILL must be 'median' or 'zero'.")
    hist = gray_histogram_from_raster(src, vv_idx, preproc, calibration_mode, scope_window=scope_window)
    return histogram_median_gray(hist)


def _resolve_locked_threshold(ckpt, checkpoint_path):
    sidecar = os.path.join(os.path.dirname(checkpoint_path), "best_threshold.json")
    if os.path.exists(sidecar):
        try:
            with open(sidecar) as f:
                rec = json.load(f)
            return float(rec["best_threshold"]), "sidecar+checkpoint"
        except Exception as e:
            print("WARNING: could not read threshold sidecar; falling back to checkpoint:", e)
    return float(ckpt.get("threshold", THRESHOLD)), "checkpoint"


def _run_tiled_inference(model, pre_src, post_src, pre_vv_idx, post_vv_idx, preproc,
                         base_window: Window, tile_size: int, overlap: int, device,
                         nodata_fill: str, tta: bool, calibration_mode: str,
                         fill_scope_window: Optional[Window] = None):
    width = int(base_window.width); height = int(base_window.height)
    base_x = int(base_window.col_off); base_y = int(base_window.row_off)
    if tile_size <= overlap:
        raise ValueError("tile_size must be larger than overlap.")
    stride = tile_size - overlap
    weight = hann_window_2d(tile_size)

    # Primary full-scene inference uses scene-level medians. Diagnostic ROI inference may pass an ROI scope explicitly.
    pre_fill = _resolve_nodata_fill(pre_src, pre_vv_idx, preproc, nodata_fill, calibration_mode, scope_window=fill_scope_window)
    post_fill = _resolve_nodata_fill(post_src, post_vv_idx, preproc, nodata_fill, calibration_mode, scope_window=fill_scope_window)
    print(f"NoData fill gray: PRE={pre_fill:.1f}, POST={post_fill:.1f} ({nodata_fill})")

    prob_accum = np.zeros((height, width), dtype=np.float32)
    weight_accum = np.zeros((height, width), dtype=np.float32)
    valid_accum = np.zeros((height, width), dtype=bool)

    def tile_starts(length):
        if length <= tile_size:
            return [0]
        starts = list(range(0, length - tile_size + 1, stride))
        last = length - tile_size
        if starts[-1] != last:
            starts.append(last)
        return starts

    y_positions = tile_starts(height); x_positions = tile_starts(width)
    skipped = 0; total_tiles = len(y_positions) * len(x_positions)

    model.eval()
    with torch.no_grad():
        for y0 in tqdm(y_positions, desc="Inference rows"):
            for x0 in x_positions:
                h = min(tile_size, height-y0); w = min(tile_size, width-x0)
                win = Window(base_x+x0, base_y+y0, w, h)
                pre_db = pre_src.read(pre_vv_idx, window=win).astype(np.float32)
                post_db = post_src.read(post_vv_idx, window=win).astype(np.float32)
                pre_db = _external_calibration_db(pre_db, preproc, calibration_mode)
                post_db = _external_calibration_db(post_db, preproc, calibration_mode)

                pre01, pre_valid = preproc.from_db(pre_db, fill_gray=pre_fill)
                post01, post_valid = preproc.from_db(post_db, fill_gray=post_fill)
                valid = pre_valid & post_valid
                if not valid.any():
                    skipped += 1
                    continue

                pre01 = _safe_pad_2d(pre01, tile_size, constant_value=pre_fill/255.0)
                post01 = _safe_pad_2d(post01, tile_size, constant_value=post_fill/255.0)
                pre_in = preproc.expand_channels(pre01); post_in = preproc.expand_channels(post01)
                assert pre_in.shape == (preproc.in_channels, tile_size, tile_size)
                assert post_in.shape == (preproc.in_channels, tile_size, tile_size)
                pre_t = torch.from_numpy(pre_in).unsqueeze(0).float().to(device)
                post_t = torch.from_numpy(post_in).unsqueeze(0).float().to(device)
                prob_tile = predict_prob_tensor(model, pre_t, post_t, use_tta=tta).squeeze().cpu().numpy()

                w_use = weight[:h, :w]
                prob_accum[y0:y0+h, x0:x0+w] += prob_tile[:h, :w] * w_use
                weight_accum[y0:y0+h, x0:x0+w] += w_use
                valid_accum[y0:y0+h, x0:x0+w] |= valid

    final_prob = np.divide(prob_accum, weight_accum, out=np.zeros_like(prob_accum), where=weight_accum > 0)
    final_prob[~valid_accum] = np.nan
    return final_prob, valid_accum, {"n_tiles_total": total_tiles, "n_tiles_skipped_fully_invalid": skipped,
                                     "pre_fill_gray": pre_fill, "post_fill_gray": post_fill}


def _projected_linear_unit_to_m(crs) -> float:
    try:
        factor = crs.linear_units_factor
        if isinstance(factor, (tuple, list)):
            return float(factor[-1])
        return float(factor)
    except Exception:
        return 1.0


def area_statistics(binary_mask: np.ndarray, valid_mask: np.ndarray, transform, crs) -> Dict[str, Any]:
    flood = (binary_mask == 1) & valid_mask
    valid = valid_mask.astype(bool)
    if crs is not None and crs.is_geographic:
        if abs(transform.b) > 1e-12 or abs(transform.d) > 1e-12:
            print("WARNING: geographic raster has rotation/shear; row-area approximation assumes a north-up grid.")
        R = 6371008.8
        dlon = math.radians(abs(transform.a))
        flood_area = 0.0; valid_area = 0.0
        row_areas = []
        for r in range(binary_mask.shape[0]):
            lat_top = (transform * (0, r))[1]
            lat_bottom = (transform * (0, r+1))[1]
            px_area = R * R * dlon * abs(math.sin(math.radians(lat_top)) - math.sin(math.radians(lat_bottom)))
            row_areas.append(px_area)
            flood_area += int(flood[r].sum()) * px_area
            valid_area += int(valid[r].sum()) * px_area
        method = "latitude-dependent spherical row area"
        pixel_area = float(np.mean(row_areas)) if row_areas else np.nan
    else:
        unit_to_m = _projected_linear_unit_to_m(crs) if crs is not None else 1.0
        pixel_area = abs(transform.a * transform.e - transform.b * transform.d) * (unit_to_m ** 2)
        flood_area = float(flood.sum()) * pixel_area
        valid_area = float(valid.sum()) * pixel_area
        method = "projected affine determinant"
    return {
        "area_method": method,
        "representative_pixel_area_m2": float(pixel_area),
        "n_flood_pixels": int(flood.sum()), "n_valid_pixels": int(valid.sum()),
        "flood_area_m2": float(flood_area), "flood_area_km2": float(flood_area / 1e6),
        "valid_area_km2": float(valid_area / 1e6),
        "flood_percent_of_valid_area": float(100.0 * flood_area / valid_area) if valid_area > 0 else np.nan,
    }


def predict_geotiff(pre_tif, post_tif, checkpoint_path, output_dir,
                     tile_size=INFERENCE_TILE_SIZE, overlap=INFERENCE_OVERLAP, device=None,
                     tta=INFERENCE_TTA, nodata_fill=NODATA_FILL, calibration_mode=EXTERNAL_CALIBRATION,
                     preprocessor_override: Optional[S1GFloodsPreprocessor] = None,
                     roi_window: Optional[Window] = None, diagnostic_fill_from_roi: bool = False):
    os.makedirs(output_dir, exist_ok=True)
    device = device or DEVICE
    model, ckpt = load_model_from_checkpoint(checkpoint_path, device)
    model.eval()
    preproc = preprocessor_override or S1GFloodsPreprocessor.from_dict(ckpt["preprocessing"])
    threshold, threshold_source = _resolve_locked_threshold(ckpt, checkpoint_path)
    print(f"Inference threshold = {threshold:.4f} ({threshold_source})")
    print("Preprocessing loaded from checkpoint" if preprocessor_override is None else "DIAGNOSTIC: preprocessing override supplied")
    print("dB→gray lo/hi used:", preproc.lo, preproc.hi)

    compat = check_pre_post_compatibility(pre_tif, post_tif)
    with rasterio.open(pre_tif) as pre_src, rasterio.open(post_tif) as post_src:
        if roi_window is None:
            base_window = Window(0, 0, pre_src.width, pre_src.height)
        else:
            base_window = Window(*map(int, [roi_window.col_off, roi_window.row_off, roi_window.width, roi_window.height]))
            if (base_window.col_off < 0 or base_window.row_off < 0 or
                base_window.col_off + base_window.width > pre_src.width or
                base_window.row_off + base_window.height > pre_src.height):
                raise ValueError(f"ROI window {base_window} falls outside raster bounds {pre_src.width}×{pre_src.height}")

        fill_scope = base_window if diagnostic_fill_from_roi else None
        final_prob, valid, tile_meta = _run_tiled_inference(
            model, pre_src, post_src, compat["pre_vv_index"], compat["post_vv_index"], preproc,
            base_window, tile_size, overlap, device, nodata_fill, tta, calibration_mode,
            fill_scope_window=fill_scope,
        )
        transform = pre_src.window_transform(base_window)
        crs = pre_src.crs

    final_binary = np.where(valid, (final_prob >= threshold).astype(np.uint8), 255).astype(np.uint8)
    calibrated = calibration_mode != "none"
    prefix = "calibrated_approximate_" if calibrated else ""
    out_meta = dict(driver="GTiff", height=final_prob.shape[0], width=final_prob.shape[1], count=1,
                    crs=crs, transform=transform, compress="lzw")

    prob_path = os.path.join(output_dir, f"{prefix}flood_probability.tif")
    with rasterio.open(prob_path, "w", dtype="float32", nodata=np.nan, **out_meta) as dst:
        dst.write(final_prob.astype(np.float32), 1)
    mask_path = os.path.join(output_dir, f"{prefix}flood_binary_mask.tif")
    with rasterio.open(mask_path, "w", dtype="uint8", nodata=255, **out_meta) as dst:
        dst.write(final_binary, 1)
    validity_path = os.path.join(output_dir, f"{prefix}validity_mask.tif")
    with rasterio.open(validity_path, "w", dtype="uint8", nodata=None, **out_meta) as dst:
        dst.write(valid.astype(np.uint8), 1)

    fig, ax = plt.subplots(figsize=(10, 10 * final_prob.shape[0] / final_prob.shape[1]))
    im = ax.imshow(np.ma.masked_invalid(final_prob), cmap="Blues", vmin=0, vmax=1)
    title = "Flood probability quicklook"
    if calibrated: title += " — CALIBRATED / APPROXIMATE"
    ax.set_title(title); ax.axis("off"); plt.colorbar(im, ax=ax, fraction=0.03)
    quicklook_path = os.path.join(output_dir, f"{prefix}quicklook.png")
    fig.savefig(quicklook_path, dpi=200, bbox_inches="tight"); plt.close(fig)

    area = area_statistics(final_binary, valid, transform, crs)
    meta_out = {
        "pre_tif": pre_tif, "post_tif": post_tif, "checkpoint": checkpoint_path,
        "model_name": ckpt.get("model_name"), "checkpoint_epoch": ckpt.get("epoch"),
        "threshold_used": threshold, "threshold_source": threshold_source,
        "preprocessing": preproc.to_dict(), "png_db_lo_used": preproc.lo, "png_db_hi_used": preproc.hi,
        "preprocessor_source": "diagnostic_override" if preprocessor_override is not None else "checkpoint",
        "external_input_is_composite_UNVERIFIED": bool(EXTERNAL_INPUT_IS_COMPOSITE),
        "nodata_fill": nodata_fill, "tile_size": tile_size, "overlap": overlap, "tta": bool(tta),
        "calibration_mode": calibration_mode,
        "calibrated_approximate": calibrated,
        "roi_window": [int(base_window.col_off), int(base_window.row_off), int(base_window.width), int(base_window.height)] if roi_window is not None else None,
        "valid_pixel_fraction": float(valid.mean()), "crs": str(crs), "transform": tuple(transform),
        **tile_meta, **area, "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
    }
    with open(os.path.join(output_dir, f"{prefix}external_inference_metadata.json"), "w") as f:
        json.dump(meta_out, f, indent=2, default=str)
    print(f"Flood area: {area['flood_area_km2']:.4f} km² ({area['flood_percent_of_valid_area']:.3f}% of valid area)")
    return prob_path, mask_path, validity_path, meta_out

print("predict_geotiff() ready. Primary inference restores preprocessing and threshold from best_model.pt.")


predict_geotiff() ready. Primary inference restores preprocessing and threshold from best_model.pt.


In [39]:
# Cell 35 — Example external inference call (Gaibandha)
# The external raster is never resized. Real VV dB is mapped into the checkpoint's S1GFloods gray domain.

if os.path.exists(EXTERNAL_PRE_TIF) and os.path.exists(EXTERNAL_POST_TIF):
    ext_out_dir = os.path.join(DIRS["inference_outputs"], "gaibandha")
    prob_path, mask_path, validity_path, ext_meta = predict_geotiff(
        EXTERNAL_PRE_TIF, EXTERNAL_POST_TIF, BEST_CKPT_PATH, ext_out_dir,
        tile_size=INFERENCE_TILE_SIZE, overlap=INFERENCE_OVERLAP, device=DEVICE,
        tta=INFERENCE_TTA, nodata_fill=NODATA_FILL, calibration_mode=EXTERNAL_CALIBRATION,
    )
    print("External inference complete:")
    print(" ", prob_path); print(" ", mask_path); print(" ", validity_path)
    print("Metadata:", json.dumps(ext_meta, indent=2, default=str))
    if EXTERNAL_CALIBRATION != "none":
        print("WARNING: these outputs are explicitly CALIBRATED / APPROXIMATE, not raw checkpoint-domain outputs.")
else:
    print("External GeoTIFFs not found. Update EXTERNAL_PRE_TIF / EXTERNAL_POST_TIF in Cell 4 and re-run this cell.")
    ext_out_dir, ext_meta = None, None
    prob_path = mask_path = validity_path = None

Inference threshold = 0.9000 (sidecar+checkpoint)
Preprocessing loaded from checkpoint
dB→gray lo/hi used: -30.0 0.0
PRE/POST compatibility PASSED: width/height/CRS/transform/resolution/bounds match; VV is available in both.
NoData fill gray: PRE=181.0, POST=173.0 (median)


Inference rows:   0%|          | 0/52 [00:00<?, ?it/s]

Flood area: 387.5685 km² (17.832% of valid area)
External inference complete:
  /kaggle/working/changeMamba_s1gfloods/inference_outputs/gaibandha/flood_probability.tif
  /kaggle/working/changeMamba_s1gfloods/inference_outputs/gaibandha/flood_binary_mask.tif
  /kaggle/working/changeMamba_s1gfloods/inference_outputs/gaibandha/validity_mask.tif
Metadata: {
  "pre_tif": "/kaggle/input/datasets/ashrifrahman/gaibandha-sar/SPARRSO/Gaibandha_S1_pre_June 8_20 2024.tif",
  "post_tif": "/kaggle/input/datasets/ashrifrahman/gaibandha-sar/SPARRSO/Gaibandha_S1_post_June 25_July 10_2024.tif",
  "checkpoint": "/kaggle/working/changeMamba_s1gfloods/checkpoints/best_model.pt",
  "model_name": "siamese_unet",
  "checkpoint_epoch": 150,
  "threshold_used": 0.9,
  "threshold_source": "sidecar+checkpoint",
  "preprocessing": {
    "lo": -30.0,
    "hi": 0.0,
    "quantize": true,
    "in_channels": 1,
    "input_mode": "1ch",
    "png_scale": 255.0,
    "expected_units": "dB",
    "band_name": "VV",
    "exp

In [40]:
# Cell 35B — MANDATORY GeoTIFF/PNG preprocessing parity acceptance test
# Converts 3 S1GFloods test PNG pairs back to dB with the checkpoint lo/hi, writes synthetic VV/VH GeoTIFFs,
# then requires deployment probabilities to match the PNG dataset path within PARITY_TOLERANCE without TTA.

PARITY_TEST_PASSED = False
parity_dir = os.path.join(DIRS["inference_outputs"], "geotiff_png_parity")
os.makedirs(parity_dir, exist_ok=True)
parity_model, parity_ckpt = load_model_from_checkpoint(BEST_CKPT_PATH, DEVICE)
parity_preproc = S1GFloodsPreprocessor.from_dict(parity_ckpt["preprocessing"])
parity_rows = []

for j, t in enumerate(SPLITS["test"][:PARITY_TEST_PAIRS]):
    pair_dir = os.path.join(parity_dir, f"pair_{j}"); os.makedirs(pair_dir, exist_ok=True)
    a_gray = read_png_gray(t["pre"]); b_gray = read_png_gray(t["post"])
    # Exact inverse of the assumed linear mapping at the PNG quantization grid.
    a_db = parity_preproc.lo + (a_gray.astype(np.float32) / 255.0) * (parity_preproc.hi - parity_preproc.lo)
    b_db = parity_preproc.lo + (b_gray.astype(np.float32) / 255.0) * (parity_preproc.hi - parity_preproc.lo)
    transform = from_origin(500000 + j*3000, 2600000, 10, 10)
    meta = dict(driver="GTiff", height=a_gray.shape[0], width=a_gray.shape[1], count=2,
                dtype="float32", crs="EPSG:32646", transform=transform)
    pre_tif = os.path.join(pair_dir, "pre_synthetic.tif"); post_tif = os.path.join(pair_dir, "post_synthetic.tif")
    for path, vv in [(pre_tif, a_db), (post_tif, b_db)]:
        with rasterio.open(path, "w", **meta) as dst:
            dst.write(vv.astype(np.float32), 1); dst.write(np.zeros_like(vv, dtype=np.float32), 2)
            dst.set_band_description(1, "VV"); dst.set_band_description(2, "VH")

    # PNG dataset path, no augmentation.
    ds = S1GFloodsChangeDataset([t], parity_preproc, spatial_aug=None, radiometric_aug=None)
    item = ds[0]
    with torch.no_grad():
        p_png = predict_prob_tensor(parity_model, item["pre"].unsqueeze(0).to(DEVICE),
                                    item["post"].unsqueeze(0).to(DEVICE), use_tta=False).squeeze().cpu().numpy()

    p_path, _, _, _ = predict_geotiff(
        pre_tif, post_tif, BEST_CKPT_PATH, os.path.join(pair_dir, "geotiff_inference"),
        tile_size=INFERENCE_TILE_SIZE, overlap=INFERENCE_OVERLAP, device=DEVICE,
        tta=False, nodata_fill="zero", calibration_mode="none",
    )
    with rasterio.open(p_path) as srcp:
        p_tif = srcp.read(1)
    max_abs_diff = float(np.nanmax(np.abs(p_png - p_tif)))
    parity_rows.append({"key": t["key"], "max_abs_probability_difference": max_abs_diff})
    assert max_abs_diff < PARITY_TOLERANCE, (
        f"GeoTIFF/PNG parity FAILED for {t['key']}: max abs diff {max_abs_diff:.6f} >= {PARITY_TOLERANCE}. "
        "Training and inference preprocessing have diverged."
    )

parity_df = pd.DataFrame(parity_rows)
parity_df.to_csv(os.path.join(parity_dir, "parity_test_results.csv"), index=False)
PARITY_TEST_PASSED = True
print(parity_df.to_string(index=False))
print(f"PARITY TEST PASSED: all max absolute probability differences < {PARITY_TOLERANCE} (TTA disabled).")


Inference threshold = 0.9000 (sidecar+checkpoint)
Preprocessing loaded from checkpoint
dB→gray lo/hi used: -30.0 0.0
PRE/POST compatibility PASSED: width/height/CRS/transform/resolution/bounds match; VV is available in both.
NoData fill gray: PRE=0.0, POST=0.0 (zero)


Inference rows:   0%|          | 0/1 [00:00<?, ?it/s]

Flood area: 0.3623 km² (5.528% of valid area)
Inference threshold = 0.9000 (sidecar+checkpoint)
Preprocessing loaded from checkpoint
dB→gray lo/hi used: -30.0 0.0
PRE/POST compatibility PASSED: width/height/CRS/transform/resolution/bounds match; VV is available in both.
NoData fill gray: PRE=0.0, POST=0.0 (zero)


Inference rows:   0%|          | 0/1 [00:00<?, ?it/s]

Flood area: 0.4596 km² (7.013% of valid area)
Inference threshold = 0.9000 (sidecar+checkpoint)
Preprocessing loaded from checkpoint
dB→gray lo/hi used: -30.0 0.0
PRE/POST compatibility PASSED: width/height/CRS/transform/resolution/bounds match; VV is available in both.
NoData fill gray: PRE=0.0, POST=0.0 (zero)


Inference rows:   0%|          | 0/1 [00:00<?, ?it/s]

Flood area: 2.8643 km² (43.706% of valid area)
                key  max_abs_probability_difference
test/image_4831.png                    5.960464e-08
test/image_4832.png                    5.960464e-08
test/image_4833.png                    5.960464e-08
PARITY TEST PASSED: all max absolute probability differences < 0.01 (TTA disabled).


In [41]:
# Cell 35C — Gaibandha radiometric sensitivity test (diagnostic only; never used to tune the chosen mapping)
# Set ROI_WINDOW=(col_off,row_off,width,height) in Cell 4. Median-fill diagnostics use the selected ROI to keep this test cheap.

if ROI_WINDOW is None:
    print("ROI_WINDOW=None — radiometric sensitivity test skipped. Set a user-chosen ROI in Cell 4 to run it.")
elif not (os.path.exists(EXTERNAL_PRE_TIF) and os.path.exists(EXTERNAL_POST_TIF)):
    print("External GeoTIFFs unavailable — sensitivity test skipped.")
else:
    roi = Window(*map(int, ROI_WINDOW))
    sens_dir = os.path.join(DIRS["inference_outputs"], "radiometric_sensitivity")
    os.makedirs(sens_dir, exist_ok=True)
    sens_rows = []; sens_masks = {}
    sens_model, sens_ckpt = load_model_from_checkpoint(BEST_CKPT_PATH, DEVICE)
    locked_threshold, _ = _resolve_locked_threshold(sens_ckpt, BEST_CKPT_PATH)

    for lo, hi in SENSITIVITY_DB_RANGES:
        for fill_mode in SENSITIVITY_NODATA_FILLS:
            pp = S1GFloodsPreprocessor(lo, hi, PNG_QUANTIZE, sens_ckpt["config"]["IN_CHANNELS"], sens_ckpt["config"]["INPUT_MODE"])
            tag = f"lo{lo:g}_hi{hi:g}_{fill_mode}".replace("-", "m").replace(".", "p")
            out_dir = os.path.join(sens_dir, tag)
            p_path, m_path, v_path, meta = predict_geotiff(
                EXTERNAL_PRE_TIF, EXTERNAL_POST_TIF, BEST_CKPT_PATH, out_dir,
                tile_size=INFERENCE_TILE_SIZE, overlap=INFERENCE_OVERLAP, device=DEVICE,
                tta=False, nodata_fill=fill_mode, calibration_mode="none",
                preprocessor_override=pp, roi_window=roi, diagnostic_fill_from_roi=True,
            )
            with rasterio.open(m_path) as sm, rasterio.open(v_path) as sv:
                mask = sm.read(1); valid = sv.read(1).astype(bool)
                a = area_statistics(mask, valid, sm.transform, sm.crs)
            sens_masks[tag] = (mask == 1, valid)
            sens_rows.append({"tag": tag, "lo": lo, "hi": hi, "nodata_fill": fill_mode,
                              "flood_area_km2": a["flood_area_km2"],
                              "flood_percent_of_valid_area": a["flood_percent_of_valid_area"],
                              "threshold": locked_threshold})

    sens_df = pd.DataFrame(sens_rows)
    sens_df.to_csv(os.path.join(sens_dir, "radiometric_sensitivity_area.csv"), index=False)
    pair_rows = []
    tags = list(sens_masks)
    for i in range(len(tags)):
        for j in range(i+1, len(tags)):
            a, va = sens_masks[tags[i]]; b, vb = sens_masks[tags[j]]
            valid = va & vb; union = ((a | b) & valid).sum(); inter = ((a & b) & valid).sum()
            iou = float(inter / union) if union else 1.0
            pair_rows.append({"setting_a": tags[i], "setting_b": tags[j], "pairwise_mask_iou": iou})
    pairwise_df = pd.DataFrame(pair_rows)
    pairwise_df.to_csv(os.path.join(sens_dir, "radiometric_sensitivity_pairwise_iou.csv"), index=False)
    print("DIAGNOSTIC ONLY — never choose lo/hi from these results to improve external appearance/performance.")
    print(sens_df.to_string(index=False))
    print(pairwise_df.to_string(index=False))


ROI_WINDOW=None — radiometric sensitivity test skipped. Set a user-chosen ROI in Cell 4 to run it.


In [42]:
# Cell 35D — Export external inference GeoTIFF outputs as EPS
# EPS is visualization-only; GeoTIFFs remain the georeferenced source of record.

from matplotlib.colors import ListedColormap

def geotiff_to_eps(tif_path, eps_path, mode="probability"):
    with rasterio.open(tif_path) as src:
        arr = src.read(1, masked=True)
    height, width = arr.shape
    fig, ax = plt.subplots(figsize=(10, 10 * height / width))
    if mode == "probability":
        ax.imshow(arr, cmap="Blues", vmin=0, vmax=1, interpolation="nearest")
    elif mode == "binary":
        ax.imshow(arr, cmap=ListedColormap(["white", "blue"]), vmin=0, vmax=1, interpolation="nearest")
    elif mode == "validity":
        ax.imshow(arr, cmap=ListedColormap(["black", "white"]), vmin=0, vmax=1, interpolation="nearest")
    else:
        raise ValueError("mode must be 'probability', 'binary', or 'validity'")
    ax.axis("off"); fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
    fig.savefig(eps_path, format="eps", bbox_inches="tight", pad_inches=0); plt.close(fig)
    print("Saved:", eps_path)

if ext_out_dir is not None and all(p is not None and os.path.exists(p) for p in [prob_path, mask_path, validity_path]):
    eps_dir = os.path.join(ext_out_dir, "eps"); os.makedirs(eps_dir, exist_ok=True)
    geotiff_to_eps(prob_path, os.path.join(eps_dir, "flood_probability.eps"), "probability")
    geotiff_to_eps(mask_path, os.path.join(eps_dir, "flood_binary_mask.eps"), "binary")
    geotiff_to_eps(validity_path, os.path.join(eps_dir, "validity_mask.eps"), "validity")
    print("EPS export complete:", eps_dir)
else:
    print("External inference outputs are unavailable — EPS export skipped.")


Saved: /kaggle/working/changeMamba_s1gfloods/inference_outputs/gaibandha/eps/flood_probability.eps
Saved: /kaggle/working/changeMamba_s1gfloods/inference_outputs/gaibandha/eps/flood_binary_mask.eps
Saved: /kaggle/working/changeMamba_s1gfloods/inference_outputs/gaibandha/eps/validity_mask.eps
EPS export complete: /kaggle/working/changeMamba_s1gfloods/inference_outputs/gaibandha/eps


In [43]:
# Cell 36 — Domain-shift report + geospatially correct external area summary
# Domain-shift diagnostics use the RAW external VV values with the checkpoint mapping (no optional two-point calibration),
# so calibration cannot hide the shift being measured.


def _hist_quantile(hist: np.ndarray, q: float) -> float:
    if hist.sum() == 0: return np.nan
    return float(np.searchsorted(np.cumsum(hist), q * hist.sum()))


def gray_hist_stats(hist: np.ndarray) -> Dict[str, float]:
    total = float(hist.sum())
    values = np.arange(256, dtype=np.float64)
    if total == 0:
        return {k: np.nan for k in ["mean","std","p0.5","p1","p5","p25","p50","p75","p95","p99","p99.5","pct_at_0","pct_at_255","pct_below_60","pct_below_99"]}
    mean = float(np.sum(values * hist) / total)
    std = float(np.sqrt(np.sum(((values-mean)**2) * hist) / total))
    out = {"mean": mean, "std": std,
           "pct_at_0": float(100*hist[0]/total), "pct_at_255": float(100*hist[255]/total),
           "pct_below_60": float(100*hist[:60].sum()/total), "pct_below_99": float(100*hist[:99].sum()/total)}
    for q in [0.005,0.01,0.05,0.25,0.50,0.75,0.95,0.99,0.995]:
        out[f"p{100*q:g}"] = _hist_quantile(hist, q)
    return out

if os.path.exists(EXTERNAL_PRE_TIF) and os.path.exists(EXTERNAL_POST_TIF):
    domain_ckpt = load_checkpoint(BEST_CKPT_PATH, "cpu")
    domain_preproc = S1GFloodsPreprocessor.from_dict(domain_ckpt["preprocessing"])
    compat = check_pre_post_compatibility(EXTERNAL_PRE_TIF, EXTERNAL_POST_TIF)
    with rasterio.open(EXTERNAL_PRE_TIF) as pre_src, rasterio.open(EXTERNAL_POST_TIF) as post_src:
        ext_pre_hist = gray_histogram_from_raster(pre_src, compat["pre_vv_index"], domain_preproc, calibration_mode="none")
        ext_post_hist = gray_histogram_from_raster(post_src, compat["post_vv_index"], domain_preproc, calibration_mode="none")

        # Figure window: use ROI_WINDOW if provided; otherwise an explicit centered window capped by config.
        if ROI_WINDOW is not None:
            fig_win = Window(*map(int, ROI_WINDOW))
        else:
            fw = min(DOMAIN_FIGURE_MAX_SIZE, pre_src.width); fh = min(DOMAIN_FIGURE_MAX_SIZE, pre_src.height)
            fig_win = Window((pre_src.width-fw)//2, (pre_src.height-fh)//2, fw, fh)
        pre_db_fig = pre_src.read(compat["pre_vv_index"], window=fig_win).astype(np.float32)
        post_db_fig = post_src.read(compat["post_vv_index"], window=fig_win).astype(np.float32)
        diff_db_fig = post_db_fig - pre_db_fig

    train_A_hist = np.asarray(TRAIN_GRAY_HISTS["A"], dtype=np.int64)
    train_B_hist = np.asarray(TRAIN_GRAY_HISTS["B"], dtype=np.int64)
    hist_map = {"S1GFloods_train_A": train_A_hist, "S1GFloods_train_B": train_B_hist,
                "Gaibandha_PRE": ext_pre_hist, "Gaibandha_POST": ext_post_hist}
    rows = []
    for name, hist in hist_map.items():
        row = {"source": name, **gray_hist_stats(hist)}
        if name == "Gaibandha_PRE":
            row["wasserstein_vs_matching_train"] = float(wasserstein_distance(np.arange(256), np.arange(256),
                                                                               u_weights=ext_pre_hist, v_weights=train_A_hist))
        elif name == "Gaibandha_POST":
            row["wasserstein_vs_matching_train"] = float(wasserstein_distance(np.arange(256), np.arange(256),
                                                                               u_weights=ext_post_hist, v_weights=train_B_hist))
        else:
            row["wasserstein_vs_matching_train"] = 0.0
        rows.append(row)
    domain_shift_df = pd.DataFrame(rows)
    domain_shift_df.to_csv(os.path.join(DIRS["inference_outputs"], "domain_shift_report.csv"), index=False)

    x = np.arange(256)
    for log_scale, suffix in [(False, "linear"), (True, "log")]:
        fig, ax = plt.subplots(figsize=(10, 6))
        for name, hist in hist_map.items():
            density = hist / max(hist.sum(), 1)
            ax.plot(x, density, label=name)
        if log_scale: ax.set_yscale("log")
        ax.set_xlabel("Mapped VV gray (0..255)"); ax.set_ylabel("Density")
        ax.set_title(f"S1GFloods vs Gaibandha gray-domain distributions ({suffix} scale)")
        ax.legend(); fig.savefig(os.path.join(DIRS["inference_outputs"], f"domain_histogram_{suffix}.png"), dpi=200, bbox_inches="tight"); plt.close(fig)

    finite_pair = np.isfinite(pre_db_fig) & np.isfinite(post_db_fig)
    if finite_pair.any():
        both = np.concatenate([pre_db_fig[finite_pair], post_db_fig[finite_pair]])
        vmin, vmax = np.percentile(both, [2, 98])
        dlim = float(np.percentile(np.abs(diff_db_fig[finite_pair]), 98))
    else:
        vmin, vmax, dlim = -30, 0, 10
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(pre_db_fig, cmap="gray", vmin=vmin, vmax=vmax); axes[0].set_title("Gaibandha PRE VV (dB)")
    axes[1].imshow(post_db_fig, cmap="gray", vmin=vmin, vmax=vmax); axes[1].set_title("Gaibandha POST VV (dB)")
    axes[2].imshow(diff_db_fig, cmap="RdBu", vmin=-max(dlim,1e-3), vmax=max(dlim,1e-3)); axes[2].set_title("POST - PRE VV (dB)")
    for ax in axes: ax.axis("off")
    fig.suptitle(f"Domain-shift inspection window {tuple(map(int,[fig_win.col_off,fig_win.row_off,fig_win.width,fig_win.height]))}")
    fig.tight_layout(); fig.savefig(os.path.join(DIRS["inference_outputs"], "gaibandha_pre_post_difference.png"), dpi=200, bbox_inches="tight"); plt.close(fig)

    train_dark = gray_hist_stats(train_A_hist)["pct_below_60"]
    ext_dark = gray_hist_stats(ext_pre_hist)["pct_below_60"]
    print(domain_shift_df.to_string(index=False))
    if train_dark > 0 and ext_dark < 0.5 * train_dark:
        print(f"WARNING: Gaibandha PRE dark-tail fraction (<gray 60) is far below training A: {ext_dark:.2f}% vs {train_dark:.2f}%. "
              "Flood recall may be under-estimated under this radiometric shift.")
    print("UNVERIFIED mapping used for this report:", (domain_preproc.lo, domain_preproc.hi))

    expected_mask = mask_path if mask_path is not None and os.path.exists(mask_path) else os.path.join(DIRS["inference_outputs"], "gaibandha", "flood_binary_mask.tif")
    expected_valid = validity_path if validity_path is not None and os.path.exists(validity_path) else os.path.join(DIRS["inference_outputs"], "gaibandha", "validity_mask.tif")
    if os.path.exists(expected_mask) and os.path.exists(expected_valid):
        with rasterio.open(expected_mask) as src_m, rasterio.open(expected_valid) as src_v:
            mask_arr = src_m.read(1); valid_arr = src_v.read(1).astype(bool)
            area_summary_dict = area_statistics(mask_arr, valid_arr, src_m.transform, src_m.crs)
        area_summary = pd.DataFrame([area_summary_dict])
        area_summary.to_csv(os.path.join(DIRS["inference_outputs"], "external_area_summary.csv"), index=False)
        print(area_summary.to_string(index=False))
    else:
        print("External prediction outputs not found; domain-shift report saved, area summary skipped.")
else:
    print("External GeoTIFFs unavailable — domain-shift report and area calculation skipped.")


PRE/POST compatibility PASSED: width/height/CRS/transform/resolution/bounds match; VV is available in both.
           source       mean       std  pct_at_0  pct_at_255  pct_below_60  pct_below_99  p0.5   p1    p5   p25   p50   p75   p95   p99  p99.5  wasserstein_vs_matching_train
S1GFloods_train_A 130.579816 62.055686  6.602232    2.643188     14.450976     26.170156   0.0  0.0   0.0  96.0 136.0 176.0 224.0 255.0  255.0                       0.000000
S1GFloods_train_B 116.780180 78.129187  9.278441    1.824886     30.760816     44.827569   0.0  0.0   0.0  47.0 119.0 189.0 230.0 255.0  255.0                       0.000000
    Gaibandha_PRE 176.849380 28.308310  0.000000    0.109992      0.002940      2.513693  84.0 89.0 113.0 169.0 181.0 193.0 214.0 232.0  239.0                      48.467536
   Gaibandha_POST 159.585611 43.789873  0.002139    0.451401      0.747596     13.715751  55.0 63.0  82.0 122.0 173.0 191.0 216.0 240.0  253.0                      46.439331
UNVERIFIED mapping use

In [44]:
# Cell 37 — Experiment manifest + environment export

def get_pip_freeze():
    try:
        out = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, timeout=60)
        return out.stdout
    except Exception as e:
        return f"pip freeze failed: {e}"

env_info = {
    "python_version": platform.python_version(), "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda, "cudnn_version": torch.backends.cudnn.version(),
    "gpu_count": N_GPUS, "gpu_names": [torch.cuda.get_device_properties(i).name for i in range(N_GPUS)] if N_GPUS else [],
    "timestamp_utc": pd.Timestamp.utcnow().isoformat(), "seed": SEED,
}
with open(os.path.join(DIRS["environment"], "environment_info.json"), "w") as f: json.dump(env_info, f, indent=2)
with open(os.path.join(DIRS["environment"], "pip_freeze.txt"), "w") as f: f.write(get_pip_freeze())

experiment_manifest = {
    "dataset": {
        "name": "S1GFloods", "data_root": DATA_ROOT, "split_source": split_source,
        "counts": {k: len(v) for k, v in SPLITS.items()},
        "label_semantics": "GT=255 / GT>127 = newly inundated flood; permanent water negative",
        "split_independence_caveat": "filenames have no event/region ID; event-level independence cannot be verified from filenames",
    },
    "model": MODEL_NAME, "input_mode": INPUT_MODE, "in_channels_per_date": IN_CHANNELS,
    "loss": LOSS_NAME, "optimizer": "AdamW", "scheduler": SCHEDULER_NAME,
    "seed": SEED, "run_seeds": RUN_SEEDS, "epochs_configured": EPOCHS,
    "epochs_run": int(hist_df["epoch"].max()) + 1 if len(hist_df) else 0,
    "best_epoch": int(BEST_CKPT["epoch"]), "learning_rate": LEARNING_RATE,
    "batch_size_per_gpu": BATCH_SIZE_PER_GPU, "accum_steps": ACCUM_STEPS,
    "normalization": PREPROCESSOR.to_dict(),
    "unverified_assumptions": {"png_db_range": [PNG_DB_LO, PNG_DB_HI], "external_input_is_composite": EXTERNAL_INPUT_IS_COMPOSITE},
    "augmentations": {"spatial": USE_AUGMENTATION, "radiometric": USE_SAR_RADIOMETRIC_AUG,
                      "dark_tail": AUG_DARK_TAIL, "gain_offset": AUG_GAIN_OFFSET, "date_offset": AUG_DATE_OFFSET, "blur": AUG_BLUR},
    "threshold": THRESHOLD, "inference_tta": INFERENCE_TTA,
    "test_metrics": {k: (float(v) if isinstance(v, (int,float,np.floating,np.integer)) else v) for k,v in global_metrics.items()},
    "checkpoint": BEST_CKPT_PATH, "software_versions": env_info, "gpu": env_info["gpu_names"],
    "training_strategy": "DDP" if EFFECTIVE_DDP else ("DataParallel" if EFFECTIVE_DP else "single-GPU"),
    "runtime": complexity_df.to_dict("records")[0] if len(complexity_df) else {},
    "external_outputs_validated_claim": False,
    "validation_note": "Do not call external GeoTIFF outputs validated unless the GeoTIFF/PNG parity test passes and any supplied reference-mask evaluation also passes.",
}
with open(os.path.join(OUTPUT_ROOT, "experiment_manifest.json"), "w") as f:
    json.dump(experiment_manifest, f, indent=2, default=str)
print("Saved experiment_manifest.json and environment exports.")

KeyError: 'epoch'

In [45]:
# Cell 38 — Zip all outputs for download from Kaggle

zip_path = "/kaggle/working/changeMamba_s1gfloods_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUTPUT_ROOT):
        for file in files:
            full_path = os.path.join(root, file)
            zf.write(full_path, os.path.relpath(full_path, OUTPUT_ROOT))
size_mb = os.path.getsize(zip_path) / (1024 ** 2)
print(f"Zipped all outputs to {zip_path} ({size_mb:.1f} MB).")
for d in SUBDIRS:
    p = os.path.join(OUTPUT_ROOT, d); n = sum(len(files) for _, _, files in os.walk(p))
    print(f"  {d:<20s}: {n} files")

Zipped all outputs to /kaggle/working/changeMamba_s1gfloods_outputs.zip (473.0 MB).
  checkpoints         : 3 files
  logs                : 0 files
  metrics             : 14 files
  figures             : 4 files
  predictions         : 0 files
  qualitative         : 18 files
  configs             : 2 files
  inference_outputs   : 35 files
  environment         : 2 files
  split_audit         : 6 files


## Summary / How to use this notebook

**1. Set the data paths.** `DATA_ROOT` must point to S1GFloods containing `train/`, `val/`, `test/`, each with `A/`, `B/`, `GT/`. Set `EXTERNAL_PRE_TIF` and `EXTERNAL_POST_TIF` to the real Sentinel-1 GeoTIFFs. The notebook never creates a random split and never resizes mismatched rasters.

**2. Input mode.** Keep `INPUT_MODE="1ch"` for the default single-VV model trained from scratch. Set `INPUT_MODE="3ch_replicated"` when running the explicit 3-channel replication ablation, especially if loading ImageNet-pretrained VMamba weights. `IN_CHANNELS` is derived automatically and stored in the checkpoint configuration.

**3. UNVERIFIED dB→gray mapping.** `PNG_DB_LO=-30`, `PNG_DB_HI=0` is an inferred assumption, not an author-confirmed S1GFloods fact. It is stored in `best_model.pt` and restored during GeoTIFF inference. Do not tune it on Gaibandha to improve apparent results; use the dedicated sensitivity cell instead.

**4. Both-zero masking.** `MASK_BOTH_ZERO=False` is the default because zero values can be legitimate dark/permanent-water negatives. Enable it only if `split_audit/both_zero_border_audit.csv` and the examples demonstrate straight-edged footprint NoData.

**5. Training.** Default training is 100 epochs, LR `1e-4`, `bce_dice`, AMP on, `BATCH_SIZE_PER_GPU=4`, and gradient accumulation. If the measured positive fraction is below 10%, the notebook prints a recommendation for `focal_tversky` / `bce_tversky` but does not silently change the loss.

**6. Threshold and TTA.** Checkpoint selection remains validation-IoU based. The validation-only threshold search now writes its chosen threshold back into `best_model.pt`. `INFERENCE_TTA=False` by default; when enabled, the same rotation/flip probability averaging is used for validation/test collection and GeoTIFF inference.

**7. Real GeoTIFF inference.** `predict_geotiff(...)` reads VV by band description, restores preprocessing from the checkpoint, fills invalid footprint pixels with a scene-level median gray (or zero), performs 256/128 Hann-blended window inference, and writes probability, binary, validity, quicklook, and metadata outputs without changing the source grid.

**8. Acceptance tests.** Run the mandatory GeoTIFF↔PNG parity cell after training; it must achieve max absolute probability difference `<1e-2` on three S1GFloods test pairs without TTA. If a Gaibandha reference mask exists, set `EXTERNAL_REF_MASK` and re-run the optional external-evaluation cell after external inference.

**9. Sensitivity and domain shift.** Set `ROI_WINDOW=(col,row,width,height)` to run the external radiometric-sensitivity matrix over four fixed dB ranges and both NoData-fill modes. The domain-shift cell reports gray-level distribution differences and explicitly warns when Gaibandha lacks the training dark tail, because external flood recall may then be underestimated.

**10. Reporting language.** S1GFloods GT is treated as **newly inundated flood change**, not generic post-event water. External outputs must not be described as validated unless the parity test passes and, when a reference mask is supplied, the reference-mask evaluation also passes.